In [1]:
import os, sys, json, time, gc, hashlib, platform, warnings, zipfile, math, importlib
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import scipy, sklearn, joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

# =============================== CONFIGURATION ===============================
SMOKE_TEST = bool(int(os.environ.get("M2_SMOKE_TEST", "0")))   # True -> tiny synthetic data, code check only
SEED = 42
STUDENT_FRACTION = float(os.environ.get("M2_STUDENT_FRACTION", "1.0"))   # <1.0 keeps a deterministic random subset of STUDENTS (full histories) for quick trial runs
HGB_TRAIN_ROWS = 3_000_000    # row-sample of the TRAIN split used to fit LR / HistGB (features are already past-only)
RUN_LOOSE_SENSITIVITY = True  # also fit the teammate's "loose" variant, ONLY to quantify ordering sensitivity (never selectable)
TUNE_HGB = True               # compare two HistGB configs on validation
BOOTSTRAP_B = 300
KAGGLE_DEFAULT = "/kaggle/input/datasets/junyiacademy/learning-activity-public-dataset-by-junyi-academy"

if SMOKE_TEST:
    HGB_TRAIN_ROWS, BOOTSTRAP_B = 150_000, 100

WORK = Path(os.environ.get("M2_WORK_DIR") or ("/kaggle/working" if os.path.isdir("/kaggle/working") and not SMOKE_TEST
                                               else ("./member2_smoke_outputs" if SMOKE_TEST else "./member2_outputs")))
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
DATA_SOURCE = "synthetic-smoke-test" if SMOKE_TEST else "junyi-real"
rng_global = np.random.default_rng(SEED)

_t0 = time.time()
def tick(msg): print(f"[{time.time()-_t0:7.1f}s] {msg}", flush=True)

print("DATA SOURCE :", DATA_SOURCE, "  <-- must be 'junyi-real' for the final submission run")
print("Work dir    :", WORK.resolve())
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__)

DATA SOURCE : junyi-real   <-- must be 'junyi-real' for the final submission run
Work dir    : /kaggle/working
Python 3.13.15 | numpy 2.1.3 | pandas 2.3.3 | scikit-learn 1.6.1 | scipy 1.16.3


In [2]:
# ------------------ synthetic data (ONLY used when SMOKE_TEST=True) ------------------
def make_synthetic_junyi(out_dir, n_students=2500, n_exercises=140, n_days=120, seed=0):
    """Writes Log_Problem.csv / Info_Content.csv with the same columns/quirks as the Junyi files."""
    rng = np.random.default_rng(seed)
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ucids = np.array([f"ex{i:04d}=" for i in range(n_exercises)])
    concept = rng.integers(0, 28, n_exercises)
    diff_lvl = rng.integers(0, 3, n_exercises)
    content = pd.DataFrame({
        "ucid": ucids, "content_pretty_name": [f"Exercise {i}" for i in range(n_exercises)],
        "difficulty": np.array(["easy", "normal", "hard"], dtype=object)[diff_lvl],
        "learning_stage": rng.choice(["elementary", "junior", "senior"], n_exercises),
        "level1_id": 1, "level2_id": concept // 9, "level3_id": concept // 3, "level4_id": concept.astype(float)})
    content.loc[rng.random(n_exercises) < 0.05, "level4_id"] = np.nan      # exercises with no concept
    content.loc[rng.random(n_exercises) < 0.03, "difficulty"] = np.nan     # missing metadata
    ex_hard = diff_lvl * 0.7 + rng.normal(0, 0.3, n_exercises)

    rows = []
    t_start = pd.Timestamp("2018-08-01", tz="UTC")
    for s in range(n_students):
        uid = f"stu{s:05d}" + "x" * 30
        theta = rng.normal(0.8, 0.9)
        skill = np.zeros(28)
        day = rng.integers(0, n_days - 5)
        for _ in range(rng.integers(2, 22)):
            day = min(day + rng.integers(0, 8), n_days - 1)
            slot = int(rng.integers(0, 48))
            ts = t_start + pd.Timedelta(days=int(day), minutes=30 * slot)
            for _ in range(int(rng.integers(1, 3))):                 # 1-2 exercises in the same 30-min bucket
                e = int(rng.integers(0, n_exercises)); c = concept[e]
                for rep in range(1, int(rng.integers(1, 3)) + 1):
                    for pn in range(1, int(rng.integers(3, 9)) + 1):
                        p = 1 / (1 + np.exp(-(theta + skill[c] - ex_hard[e])))
                        yv = rng.random() < p
                        skill[c] += 0.08 if yv else 0.02
                        rows.append((ts, uid, ucids[e], pn, rep, yv))
    log = pd.DataFrame(rows, columns=["timestamp_TW", "uuid", "ucid", "problem_number",
                                      "exercise_problem_repeat_session", "is_correct"])
    log["timestamp_TW"] = log["timestamp_TW"].dt.strftime("%Y-%m-%d %H:%M:%S UTC")
    log["is_correct"] = log["is_correct"].map({True: "True", False: "False"})
    log["total_sec_taken"] = rng.integers(3, 300, len(log)).astype(float)
    log["total_attempt_cnt"] = rng.integers(1, 4, len(log)).astype(float)
    log["used_hint_cnt"] = rng.integers(0, 3, len(log)).astype(float)
    bad = rng.random(len(log)) < 0.002                                  # rows that must be removed by cleaning
    log.loc[bad, "is_correct"] = np.nan
    log = log.sample(frac=1.0, random_state=seed).reset_index(drop=True)  # real file is not time-sorted
    log.to_csv(out_dir / "Log_Problem.csv", index=False)
    content.to_csv(out_dir / "Info_Content.csv", index=False)
    return out_dir

# ------------------ locate the dataset ------------------
def find_dataset_dir():
    if SMOKE_TEST:
        return make_synthetic_junyi(WORK / "synthetic_input")
    if os.path.exists(f"{KAGGLE_DEFAULT}/Log_Problem.csv"):
        return Path(KAGGLE_DEFAULT)
    for root, _, files in os.walk("/kaggle/input"):
        if "Log_Problem.csv" in files and "Info_Content.csv" in files:
            return Path(root)
    raise FileNotFoundError("Log_Problem.csv / Info_Content.csv not found. Attach the Junyi dataset to this notebook.")

DATASET_PATH = find_dataset_dir()
LOG_PATH, CONTENT_PATH = DATASET_PATH / "Log_Problem.csv", DATASET_PATH / "Info_Content.csv"
print("Dataset dir:", DATASET_PATH)
print("Files:", sorted(p.name for p in DATASET_PATH.iterdir()))

Dataset dir: /kaggle/input/datasets/junyiacademy/learning-activity-public-dataset-by-junyi-academy
Files: ['Info_Content.csv', 'Info_UserData.csv', 'Log_Problem.csv']


## 1. Load and clean the interaction log

Same cleaning rules as the earlier notebook (drop rows missing any essential field, parse the timestamp, parse `is_correct`, drop blank ids) so the row count is directly comparable with the previous run (**16,217,311** clean rows on the full data).

Only columns that are known **before the student answers** are kept as potential inputs. `total_sec_taken`, `total_attempt_cnt`, `used_hint_cnt` describe the *current* attempt and are only known after it, so they are used for cleaning parity but **never as features**.

In [3]:
CHUNK = 500_000
REQUIRED = ["timestamp_TW", "uuid", "ucid", "is_correct", "total_sec_taken", "total_attempt_cnt", "used_hint_cnt"]
USECOLS = REQUIRED + ["problem_number", "exercise_problem_repeat_session"]
POST_ANSWER_COLUMNS = ["total_sec_taken", "total_attempt_cnt", "used_hint_cnt"]   # known only AFTER answering -> never features

class IncrementalEncoder:
    """Maps strings to stable int codes across chunks (keeps memory low for 16M rows)."""
    def __init__(self): self.index = pd.Index([], dtype=object)
    def encode(self, values):
        codes, uniq = pd.factorize(np.asarray(values, dtype=object))
        uniq = pd.Index(uniq, dtype=object)
        new = uniq.difference(self.index, sort=False)
        if len(new): self.index = self.index.append(new)
        return self.index.get_indexer(uniq)[codes].astype("int32")

stu_enc, ex_enc = IncrementalEncoder(), IncrementalEncoder()
parts, total_rows = [], 0
for chunk in pd.read_csv(LOG_PATH, usecols=USECOLS, chunksize=CHUNK):
    total_rows += len(chunk)
    chunk = chunk.dropna(subset=REQUIRED)
    ts = pd.to_datetime(chunk["timestamp_TW"], errors="coerce", utc=True)
    ic = chunk["is_correct"].astype("string").str.lower().map({"true": True, "false": False})
    ok = (ts.notna() & ic.notna()
          & (chunk["uuid"].astype(str).str.strip() != "") & (chunk["ucid"].astype(str).str.strip() != "")).to_numpy()
    chunk, ts, ic = chunk[ok], ts[ok], ic[ok]
    if STUDENT_FRACTION < 1.0:       # deterministic student-level subsample (keeps whole histories)
        h = pd.util.hash_pandas_object(chunk["uuid"].astype(str), index=False).to_numpy() % 10_000
        keep = h < int(STUDENT_FRACTION * 10_000)
        chunk, ts, ic = chunk[keep], ts[keep], ic[keep]
    if len(chunk) == 0:
        continue
    parts.append(pd.DataFrame({
        "student": stu_enc.encode(chunk["uuid"].astype(str).to_numpy()),
        "exercise": ex_enc.encode(chunk["ucid"].astype(str).to_numpy()),
        "ts": ts.dt.tz_convert(None).to_numpy().astype("datetime64[s]").astype("int64"),
        "y": ic.astype("int8").to_numpy(),
        "pn": pd.to_numeric(chunk["problem_number"], errors="coerce").fillna(0).astype("int32").to_numpy(),
        "sess": pd.to_numeric(chunk["exercise_problem_repeat_session"], errors="coerce").fillna(0).astype("int32").to_numpy(),
    }))
raw = pd.concat(parts, ignore_index=True); del parts; gc.collect()
raw["rowid"] = np.arange(len(raw), dtype="int32")      # final deterministic tie-break
uuid_values, ucid_values = stu_enc.index.to_numpy(), ex_enc.index.to_numpy()

print("Original rows processed :", f"{total_rows:,}")
print("Clean rows kept         :", f"{len(raw):,}   (removed {total_rows-len(raw):,})")
print("Unique students         :", f"{raw['student'].nunique():,}")
print("Unique exercises in log :", f"{raw['exercise'].nunique():,}")
print("Correct-answer rate     :", round(raw['y'].mean(), 4))
tick("data loaded")

Original rows processed : 16,217,311
Clean rows kept         : 16,217,311   (removed 0)
Unique students         : 72,758
Unique exercises in log : 1,326
Correct-answer rate     : 0.7037
[  169.5s] data loaded


In [4]:
SMOKE_TEST = bool(int(os.environ.get("M2_SMOKE_TEST", "0")))
STUDENT_FRACTION = float(os.environ.get("M2_STUDENT_FRACTION", "1.0"))
HGB_TRAIN_ROWS = 3_000_000

## 2. Exercise metadata, concept proxy, and the chronological split

* **Concept proxy** = `level4_id` (as in the earlier notebook). An exercise with no `level4_id` becomes its **own** pseudo-concept (`EX:<ucid>`) instead of being merged into a shared "unknown" bucket or silently getting code `-1`.
* The exercise table covers **all** exercises in `Info_Content.csv` (not only those seen in the log) so Member 3 can score exercises a student has never attempted.
* Unknown `difficulty` / `learning_stage` stay **missing (NaN)**; models and the predictor handle them explicitly.

In [5]:
content = pd.read_csv(CONTENT_PATH)
for c in ["ucid", "difficulty", "learning_stage", "level4_id"]:
    if c not in content.columns: content[c] = np.nan
content = content[["ucid", "difficulty", "learning_stage", "level4_id"]].copy()
content["ucid"] = content["ucid"].astype(str)
content = content.drop_duplicates("ucid")

universe = pd.Index(ucid_values).append(pd.Index(content["ucid"]).difference(pd.Index(ucid_values), sort=False))
ex_table = content.set_index("ucid").reindex(universe).rename_axis("ucid").reset_index()
n_ex = len(ex_table)

concept_label = np.where(ex_table["level4_id"].notna(),
                         "L4:" + ex_table["level4_id"].astype(str), "EX:" + ex_table["ucid"].astype(str)).astype(object)
concept_code_by_ex, concept_labels = pd.factorize(concept_label)
concept_code_by_ex = concept_code_by_ex.astype("int32")
n_concepts = len(concept_labels)

def code_levels(series):
    codes, levels = pd.factorize(series, sort=True)
    codes = codes.astype("float32"); codes[codes < 0] = np.nan
    return codes, [str(v) for v in levels]
difficulty_code_by_ex, difficulty_levels = code_levels(ex_table["difficulty"])
stage_code_by_ex, stage_levels = code_levels(ex_table["learning_stage"])

raw["concept"] = concept_code_by_ex[raw["exercise"].to_numpy()]
in_content = pd.Index(ucid_values).isin(content["ucid"])
print("Exercises in universe        :", n_ex, f"(log: {len(ucid_values)}, content-only: {n_ex-len(ucid_values)})")
print("Concept proxies (level4_id)  :", n_concepts, "| exercises without level4_id:", int((~ex_table['level4_id'].notna()).sum()))
print("Difficulty levels            :", difficulty_levels, "| stages:", stage_levels)
print("Log rows whose exercise has NO metadata row:", f"{(~in_content[raw['exercise'].to_numpy()]).mean():.4%}")
print("Log rows with missing difficulty            :", f"{np.isnan(difficulty_code_by_ex[raw['exercise'].to_numpy()]).mean():.4%}")

Exercises in universe        : 1330 (log: 1326, content-only: 4)
Concept proxies (level4_id)  : 171 | exercises without level4_id: 0
Difficulty levels            : ['easy', 'hard', 'normal', 'unset'] | stages: ['elementary', 'junior', 'senior']
Log rows whose exercise has NO metadata row: 0.0000%
Log rows with missing difficulty            : 0.0000%


In [6]:
# ------------- timestamp resolution + chronological split (by timestamp VALUE) -------------
ts_all = raw["ts"].to_numpy()
bucket_seconds = int(np.gcd.reduce(np.unique(ts_all)))
print(f"Timestamp resolution inferred from data: {bucket_seconds} s  ({bucket_seconds/60:.1f} min)  "
      "-> rows sharing a timestamp are NOT time-ordered relative to each other.")

t70, t85 = (int(v) for v in np.quantile(ts_all, [0.70, 0.85]))
split = np.where(ts_all <= t70, 0, np.where(ts_all <= t85, 1, 2)).astype("int8")
raw["split"] = split
SPLIT_NAMES = {0: "train", 1: "val", 2: "test"}
fmt = lambda s: dt.datetime.fromtimestamp(int(s), dt.timezone.utc).strftime("%Y-%m-%d %H:%M")

train_students = set(np.unique(raw.loc[raw.split == 0, "student"].to_numpy()))
rows = []
for k, name in SPLIT_NAMES.items():
    m = split == k
    stu = raw.loc[m, "student"].to_numpy()
    rows.append({"split": name, "rows": int(m.sum()), "share": m.mean(), "students": int(len(np.unique(stu))),
                 "start_utc": fmt(ts_all[m].min()), "end_utc": fmt(ts_all[m].max()),
                 "correct_rate": raw.loc[m, "y"].mean(),
                 "rows_from_unseen_students": 0.0 if k == 0 else float(np.mean(~np.isin(stu, list(train_students))))})
split_summary = pd.DataFrame(rows)
display(split_summary.round(4))
print("\nNOTE: the correct-answer rate DRIFTS between periods. Accuracy numbers from different periods are therefore NOT comparable;")
print("      every model must be judged against the majority baseline measured on the SAME rows.")

Timestamp resolution inferred from data: 900 s  (15.0 min)  -> rows sharing a timestamp are NOT time-ordered relative to each other.


,split,rows,share,students,start_utc,end_utc,correct_rate,rows_from_unseen_students
0,train,11352490,0.70,57117,2018-08-01 07:45,2019-04-16 08:15,0.7081,0.0000
1,val,2432488,0.15,23694,2019-04-16 08:30,2019-06-04 08:15,0.6800,0.1591
2,test,2432333,0.15,23270,2019-06-04 08:30,2019-08-01 00:00,0.7071,0.5260



NOTE: the correct-answer rate DRIFTS between periods. Accuracy numbers from different periods are therefore NOT comparable;
      every model must be judged against the majority baseline measured on the SAME rows.


## 3. History features — one function for every row (Priority 1)

For each interaction the model sees **only that student's earlier answers**. Rows are ordered by `(student, timestamp, repeat_session, problem_number, row id)`.

**Why "strict"?** The log's timestamps are coarse (see resolution above), so many rows share a timestamp and their true order is unknown. A row therefore uses only answers with a **strictly earlier timestamp** — answers inside the same timestamp bucket (possibly "future" in reality) are never visible. This is the conservative, defensible choice; the teammate's `*_strict` columns implement the same idea and this notebook builds them directly.

The *same* function (and the same code path) produces features for train, validation and test rows: validation/test rows start from the student's accumulated training history and keep updating it as observed answers arrive, exactly as the audit prescribes. The previous Logistic Regression instead used a **static** training-period summary for every validation row while the training rows used cumulative history — that inconsistency is removed here.

Features: student attempts/accuracy, previous answer, accuracy of last 10 answers, same-concept attempts/accuracy, same-exercise attempts/accuracy, minutes since previous activity, difficulty/stage/concept metadata, and train-only exercise/concept difficulty (target encoding, §3.2).

In [7]:
HISTORY_FEATURES = ["stu_attempts", "stu_acc", "prev_correct", "recent10_acc",
                    "concept_attempts", "concept_acc", "exercise_attempts", "exercise_acc", "gap_minutes"]
ENC_FEATURES = ["exercise_enc", "concept_enc"]
CAT_FEATURES = ["difficulty_code", "stage_code", "concept_cat"]
FEATURES = HISTORY_FEATURES + ENC_FEATURES + CAT_FEATURES

def _scope(key, ts, sess, pn, rowid, y, strict):
    """Exclusive running counts of the rows that share `key`, ordered chronologically.
    strict=True : count only rows with a strictly earlier timestamp (all rows of one timestamp share the value of the group's first row)
    strict=False: count every earlier row in sort order (the teammate's 'loose' variant; used only for the sensitivity check)."""
    n = len(key)
    order = np.lexsort((rowid, pn, sess, ts, key))            # last key = primary
    key_s, ts_s, y_s = key[order], ts[order], y[order].astype(np.int64)
    idx = np.arange(n)
    new_key = np.empty(n, dtype=bool); new_key[0] = True
    np.not_equal(key_s[1:], key_s[:-1], out=new_key[1:])
    start = np.maximum.accumulate(np.where(new_key, idx, 0))
    if strict:
        new_grp = new_key.copy(); new_grp[1:] |= ts_s[1:] != ts_s[:-1]
        first = np.maximum.accumulate(np.where(new_grp, idx, 0))
    else:
        first = idx
    E = np.cumsum(y_s) - y_s                                   # exclusive running sum of y
    return dict(order=order, key_s=key_s, ts_s=ts_s, y_s=y_s, first=first, start=start, E=E,
                att=first - start, cor=E[first] - E[start])

def _put(order, vals):
    out = np.empty(len(order), dtype=np.float32); out[order] = vals; return out

def _acc(att, cor):
    return np.where(att > 0, cor / np.maximum(att, 1), np.nan)

def build_history_features(student, concept, exercise, ts, y, sess, pn, rowid, n_concepts, n_exercises, strict=True):
    """Returns (dict of float32 arrays in ORIGINAL row order, context needed by BKT)."""
    student = student.astype(np.int64)
    out = {}
    # ---- student level
    s = _scope(student, ts, sess, pn, rowid, y, strict)
    att, first, E, y_s, ts_s = s["att"], s["first"], s["E"], s["y_s"], s["ts_s"]
    has = att > 0
    prev_idx = np.where(has, first - 1, 0)
    n10 = np.minimum(att, 10)
    out["stu_attempts"] = _put(s["order"], att)
    out["stu_acc"] = _put(s["order"], _acc(att, s["cor"]))
    out["prev_correct"] = _put(s["order"], np.where(has, y_s[prev_idx], np.nan))
    out["recent10_acc"] = _put(s["order"], np.where(n10 > 0, (E[first] - E[first - n10]) / np.maximum(n10, 1), np.nan))
    out["gap_minutes"] = _put(s["order"], np.where(has, (ts_s[first] - ts_s[prev_idx]) / 60.0, np.nan))
    del s
    # ---- student x concept   (also gives the ordering BKT needs)
    ck = student * n_concepts + concept
    s = _scope(ck, ts, sess, pn, rowid, y, strict)
    out["concept_attempts"] = _put(s["order"], s["att"])
    out["concept_acc"] = _put(s["order"], _acc(s["att"], s["cor"]))
    ctx = dict(order=s["order"], key_s=s["key_s"], ts_s=s["ts_s"], y_s=s["y_s"]); del s
    # ---- student x exercise
    ek = student * n_exercises + exercise
    s = _scope(ek, ts, sess, pn, rowid, y, strict)
    out["exercise_attempts"] = _put(s["order"], s["att"])
    out["exercise_acc"] = _put(s["order"], _acc(s["att"], s["cor"]))
    del s
    return out, ctx

hist, bkt_ctx = build_history_features(
    raw["student"].to_numpy(), raw["concept"].to_numpy(), raw["exercise"].to_numpy().astype(np.int64),
    raw["ts"].to_numpy(), raw["y"].to_numpy(), raw["sess"].to_numpy(), raw["pn"].to_numpy(), raw["rowid"].to_numpy(),
    n_concepts, n_ex, strict=True)
tick("strict history features built for ALL rows (train+val+test) with one function")
print({k: f"{np.isnan(v).mean():.3%} NaN" for k, v in hist.items() if np.isnan(v).any()})

[  194.3s] strict history features built for ALL rows (train+val+test) with one function
{'stu_acc': '3.101% NaN', 'prev_correct': '3.101% NaN', 'recent10_acc': '3.101% NaN', 'gap_minutes': '3.101% NaN', 'concept_acc': '20.793% NaN', 'exercise_acc': '52.176% NaN'}


### 3.2 Exercise / concept difficulty (target encoding) — fitted on the training period only

`exercise_enc` and `concept_enc` are smoothed training-period correct rates (prior strength 20).
**Fix:** the earlier version averaged *all* training rows, so every **training** row contained its own label inside its own feature (mild target leakage that only affects training). Here training rows get an **out-of-fold** value (5 folds, split by student); validation/test rows get the full-training value. Unknown exercises/concepts fall back to the training prior.

In [8]:
def oof_target_encoding(key, y, is_train, fold, n_keys, n_folds=5, prior=0.7, m=20.0):
    """Smoothed per-key correct rate from TRAIN rows. Train rows use out-of-fold values (own fold excluded)."""
    key = key.astype(np.int64)
    tr = is_train
    S = np.bincount(key[tr], weights=y[tr], minlength=n_keys)
    N = np.bincount(key[tr], minlength=n_keys).astype(float)
    kf = key[tr] * n_folds + fold[tr]
    Skf = np.bincount(kf, weights=y[tr], minlength=n_keys * n_folds).reshape(n_keys, n_folds)
    Nkf = np.bincount(kf, minlength=n_keys * n_folds).reshape(n_keys, n_folds).astype(float)
    table = (S + m * prior) / (N + m)
    enc = table[key]
    k_tr, f_tr = key[tr], fold[tr]
    enc_tr = (S[k_tr] - Skf[k_tr, f_tr] + m * prior) / (N[k_tr] - Nkf[k_tr, f_tr] + m)
    enc = enc.copy(); enc[tr] = enc_tr
    return enc.astype(np.float32), table

is_train = split == 0
PRIOR = float(raw.loc[is_train, "y"].mean())
N_FOLDS, SMOOTH_M = 5, 20.0
fold_of_student = np.random.default_rng(SEED).integers(0, N_FOLDS, int(raw["student"].max()) + 1)
fold = fold_of_student[raw["student"].to_numpy()]
yv = raw["y"].to_numpy().astype(np.float64)

exercise_enc_rows, exercise_enc_table = oof_target_encoding(raw["exercise"].to_numpy(), yv, is_train, fold, n_ex, N_FOLDS, PRIOR, SMOOTH_M)
concept_enc_rows,  concept_enc_table  = oof_target_encoding(raw["concept"].to_numpy(),  yv, is_train, fold, n_concepts, N_FOLDS, PRIOR, SMOOTH_M)

# concept as a capped categorical for HistGB / LR (HistGB allows <=255 categories): top-254 by TRAIN frequency, rest -> "other"
MAX_CAT = 254
cnt = np.bincount(raw.loc[is_train, "concept"].to_numpy(), minlength=n_concepts)
rank = np.argsort(-cnt, kind="stable")[:MAX_CAT]; rank = rank[cnt[rank] > 0]
concept_cat_map = np.full(n_concepts, MAX_CAT, dtype=np.float32); concept_cat_map[rank] = np.arange(len(rank))

feat = pd.DataFrame({**hist,
    "exercise_enc": exercise_enc_rows, "concept_enc": concept_enc_rows,
    "difficulty_code": difficulty_code_by_ex[raw["exercise"].to_numpy()],
    "stage_code": stage_code_by_ex[raw["exercise"].to_numpy()],
    "concept_cat": concept_cat_map[raw["concept"].to_numpy()]})[FEATURES].astype("float32")
y_all = raw["y"].to_numpy().astype(np.int8)
student_all = raw["student"].to_numpy()

# exercise lookup table = what Member 3's interface uses at prediction time (full-train encodings, no OOF)
ex_lookup = pd.DataFrame({
    "ucid": ex_table["ucid"], "difficulty": ex_table["difficulty"], "learning_stage": ex_table["learning_stage"],
    "concept_id_raw": ex_table["level4_id"], "concept_label": concept_labels[concept_code_by_ex],
    "concept_code": concept_code_by_ex, "concept_cat": concept_cat_map[concept_code_by_ex],
    "difficulty_code": difficulty_code_by_ex, "stage_code": stage_code_by_ex,
    "exercise_enc": exercise_enc_table.astype("float32"),
    "concept_enc": concept_enc_table[concept_code_by_ex].astype("float32")})
print("Feature matrix:", feat.shape, f"({feat.memory_usage().sum()/1e9:.2f} GB)   concept categories used by HistGB/LR:", int(concept_cat_map[concept_cat_map<MAX_CAT].max())+1 if (concept_cat_map<MAX_CAT).any() else 0)
print("Train prior P(correct) =", round(PRIOR, 4))
display(feat.describe().T[["count", "mean", "std", "min", "max"]].round(3))

Feature matrix: (16217311, 14) (0.91 GB)   concept categories used by HistGB/LR: 171
Train prior P(correct) = 0.7081


,count,mean,std,min,max
stu_attempts,16217311.0,592.848,845.852,0.000,11809.000
stu_acc,15714413.0,0.708,0.171,0.000,1.000
prev_correct,15714413.0,0.772,0.391,0.000,1.000
recent10_acc,15714413.0,0.717,0.230,0.000,1.000
concept_attempts,16217311.0,36.141,53.336,0.000,1777.000
concept_acc,12845258.0,0.692,0.233,0.000,1.000
exercise_attempts,16217311.0,5.869,17.824,0.000,1696.000
exercise_acc,7755825.0,0.590,0.334,0.000,1.000
gap_minutes,15714413.0,3313.453,15410.469,15.000,499530.000
exercise_enc,16217311.0,0.705,0.144,0.161,0.965


## 4. Leakage and consistency audit (Priority 2) — automated, with evidence

"Strict" in a column name does not prove anything, so the claims are tested:

1. **Split order** – every train timestamp is earlier than every validation timestamp, which is earlier than every test timestamp.
2. **Forbidden inputs** – no post-answer column and no copy of the label is a feature.
3. **Independent recomputation** – a naive, loop-based re-implementation (different code, written from the definition) must reproduce the vectorised features exactly on a random set of students, and a student's features must not depend on other students.
4. **Label-flip (perturbation) test** – flip the label of one interaction and rebuild the features: nothing at or before that timestamp (including the row itself and every row in the same timestamp bucket) may change; later rows may. This directly proves the answer being predicted cannot reach its own features.
5. **Target-encoding test** – flipping a training label must not change the out-of-fold encoding of any training row in the same fold (including the row itself).
6. **Same-semantics check** – features have identical meaning/NaN behaviour in train, validation and test.

All results are stored and printed again in the final checklist. *(Results are never asserted — only the implementation is.)*

In [9]:
AUDIT = {}
def record(name, passed, detail=""):
    AUDIT[name] = {"passed": bool(passed), "detail": detail}
    print(("PASS  " if passed else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

# 1) split order ------------------------------------------------------------
tr_max, va_min, va_max, te_min = (ts_all[split == 0].max(), ts_all[split == 1].min(),
                                  ts_all[split == 1].max(), ts_all[split == 2].min())
record("Chronological split: train < validation < test (no shared timestamp)", tr_max < va_min and va_max < te_min,
       f"train ends {fmt(tr_max)}, val {fmt(va_min)}..{fmt(va_max)}, test starts {fmt(te_min)}")

# 2) forbidden inputs ---------------------------------------------------------
forbidden = set(POST_ANSWER_COLUMNS) | {"is_correct", "y", "timestamp_TW", "uuid", "ucid"}
rs = np.random.default_rng(1); samp = rs.choice(len(feat), size=min(1_000_000, len(feat)), replace=False)
max_corr = max(abs(np.corrcoef(np.nan_to_num(feat[c].to_numpy()[samp], nan=0.5), y_all[samp])[0, 1]) for c in FEATURES)
record("No post-answer column / label copy among features", not (set(FEATURES) & forbidden) and max_corr < 0.9,
       f"max |corr(feature, label)| = {max_corr:.3f}")

# 3) independent recomputation + student independence -------------------------
def naive_features(sub):
    """Loop-based definition of the strict features (only strictly-earlier timestamps of the SAME student)."""
    n = len(sub["y"]); names = HISTORY_FEATURES
    out = {k: np.full(n, np.nan) for k in names}
    for s in np.unique(sub["student"]):
        rows = np.flatnonzero(sub["student"] == s)
        rows = rows[np.lexsort((sub["rowid"][rows], sub["pn"][rows], sub["sess"][rows], sub["ts"][rows]))]
        t, yy, cc, ee = sub["ts"][rows], sub["y"][rows].astype(float), sub["concept"][rows], sub["exercise"][rows]
        for pos in range(len(rows)):
            r = rows[pos]
            pe = int(np.searchsorted(t, t[pos], side="left"))          # rows with strictly earlier timestamp
            out["stu_attempts"][r] = pe
            if pe:
                out["stu_acc"][r] = yy[:pe].mean(); out["prev_correct"][r] = yy[pe - 1]
                out["recent10_acc"][r] = yy[max(0, pe - 10):pe].mean(); out["gap_minutes"][r] = (t[pos] - t[pe - 1]) / 60.0
            mc, me = cc[:pe] == cc[pos], ee[:pe] == ee[pos]
            out["concept_attempts"][r] = mc.sum(); out["exercise_attempts"][r] = me.sum()
            if mc.any(): out["concept_acc"][r] = yy[:pe][mc].mean()
            if me.any(): out["exercise_acc"][r] = yy[:pe][me].mean()
    return out

row_counts = np.bincount(student_all)
eligible = np.flatnonzero((row_counts >= 5) & (row_counts <= 400))
if len(eligible) < 20: eligible = np.argsort(row_counts)[-min(120, len(row_counts)):]
pick = np.random.default_rng(7).choice(eligible, size=min(120, len(eligible)), replace=False)
sidx = np.flatnonzero(np.isin(student_all, pick))
sub = {k: raw[k].to_numpy()[sidx] for k in ["student", "concept", "exercise", "ts", "y", "sess", "pn", "rowid"]}

def build_subset(y_override=None):
    yy = sub["y"] if y_override is None else y_override
    f, _ = build_history_features(sub["student"], sub["concept"], sub["exercise"].astype(np.int64), sub["ts"], yy,
                                  sub["sess"], sub["pn"], sub["rowid"], n_concepts, n_ex, strict=True)
    return f

f_sub, f_naive = build_subset(), naive_features(sub)
max_err_naive = max(np.nanmax(np.abs(f_sub[k].astype(float) - f_naive[k]))
                    if np.isfinite(f_naive[k]).any() else 0.0 for k in HISTORY_FEATURES)
nan_match = all(np.array_equal(np.isnan(f_sub[k]), np.isnan(f_naive[k])) for k in HISTORY_FEATURES)
record("Vectorised features == independent naive recomputation", max_err_naive < 1e-4 and nan_match,
       f"{len(pick)} students, {len(sidx):,} rows, max abs error {max_err_naive:.2e}, NaN pattern identical: {nan_match}")
max_err_full = max(np.nanmax(np.abs(f_sub[k].astype(float) - hist[k][sidx].astype(float)))
                   if np.isfinite(hist[k][sidx]).any() else 0.0 for k in HISTORY_FEATURES)
record("A student's features do not depend on other students", max_err_full < 1e-4, f"max abs diff vs full run {max_err_full:.2e}")

# 4) label-flip (perturbation) test ------------------------------------------------
rp = np.random.default_rng(11); changed_later, trials, violations = 0, 40, 0
for r in rp.choice(len(sidx), size=trials, replace=False):
    y2 = sub["y"].copy(); y2[r] = 1 - y2[r]
    f2 = build_subset(y2)
    same_student = sub["student"] == sub["student"][r]
    not_future = (~same_student) | (sub["ts"] <= sub["ts"][r])           # rows that must be completely unaffected
    later = same_student & (sub["ts"] > sub["ts"][r])
    diff = np.zeros(len(sidx), dtype=bool)
    for k in HISTORY_FEATURES:
        a, b = f_sub[k], f2[k]
        diff |= ~((a == b) | (np.isnan(a) & np.isnan(b)))
    violations += int(diff[not_future].sum()); changed_later += int(diff[later].any())
record("Label-flip test: flipping an answer never changes its own / same-timestamp / earlier / other-student features",
       violations == 0, f"{trials} flips, violations={violations}; later rows changed in {changed_later}/{trials} flips (test has power)")

# 5) target-encoding (out-of-fold) test --------------------------------------------
tr_rows = np.flatnonzero(is_train); bad = 0
for r in np.random.default_rng(5).choice(tr_rows, size=3, replace=False):
    y2 = yv.copy(); y2[r] = 1 - y2[r]
    for key_arr, n_keys, base in [(raw["exercise"].to_numpy(), n_ex, exercise_enc_rows), (raw["concept"].to_numpy(), n_concepts, concept_enc_rows)]:
        e2, _ = oof_target_encoding(key_arr, y2, is_train, fold, n_keys, N_FOLDS, PRIOR, SMOOTH_M)
        same_fold_train = is_train & (fold == fold[r])      # val/test rows legitimately use the full-train table, so only TRAIN rows of the fold are checked
        bad += int((np.abs(e2[same_fold_train] - base[same_fold_train]) > 1e-7).sum())
record("Target encoding is out-of-fold (a training label never reaches its own fold's training features)", bad == 0, f"3 flips x 2 encoders, violations={bad}")

# 6) same semantics in every split ---------------------------------------------------
cons = (pd.DataFrame({"split": [SPLIT_NAMES[s] for s in split[samp]]}).join(feat.iloc[samp].reset_index(drop=True))
        .groupby("split").agg(**{f"{c}": (c, "mean") for c in ["stu_attempts", "stu_acc", "recent10_acc", "concept_acc", "exercise_acc", "gap_minutes", "exercise_enc"]}))
nan_rate = (pd.DataFrame({"split": [SPLIT_NAMES[s] for s in split[samp]]}).join(feat.iloc[samp].isna().reset_index(drop=True))
            .groupby("split")[["stu_acc", "concept_acc", "exercise_acc", "gap_minutes", "difficulty_code"]].mean())
display(cons.round(3)); display(nan_rate.round(4).add_suffix("_nan"))
nan_ok = bool(((feat["stu_acc"].isna()) == (feat["stu_attempts"] == 0)).all() and ((feat["concept_acc"].isna()) == (feat["concept_attempts"] == 0)).all()
              and ((feat["exercise_acc"].isna()) == (feat["exercise_attempts"] == 0)).all())
record("Same feature semantics in train / val / test (NaN only when there is no earlier answer)", nan_ok and set(np.unique(y_all)) <= {0, 1})
del hist; gc.collect()          # history arrays now live in `feat`; free the duplicate
tick("audit tests done")

PASS  Chronological split: train < validation < test (no shared timestamp)  [train ends 2019-04-16 08:15, val 2019-04-16 08:30..2019-06-04 08:15, test starts 2019-06-04 08:30]
PASS  No post-answer column / label copy among features  [max |corr(feature, label)| = 0.292]
PASS  Vectorised features == independent naive recomputation  [120 students, 11,033 rows, max abs error 2.97e-08, NaN pattern identical: True]
PASS  A student's features do not depend on other students  [max abs diff vs full run 0.00e+00]
PASS  Label-flip test: flipping an answer never changes its own / same-timestamp / earlier / other-student features  [40 flips, violations=0; later rows changed in 37/40 flips (test has power)]
PASS  Target encoding is out-of-fold (a training label never reaches its own fold's training features)  [3 flips x 2 encoders, violations=0]


,stu_attempts,stu_acc,recent10_acc,concept_acc,exercise_acc,gap_minutes,exercise_enc
split,,,,,,,
test,752.398987,0.711,0.722,0.697,0.601,4030.104004,0.707
train,497.990997,0.712,0.721,0.696,0.593,2844.944092,0.708
val,879.672974,0.687,0.697,0.670,0.566,4642.441895,0.684


,stu_acc_nan,concept_acc_nan,exercise_acc_nan,gap_minutes_nan,difficulty_code_nan
split,,,,,
test,0.0281,0.1997,0.5142,0.0281,0.0
train,0.0339,0.2116,0.5253,0.0339,0.0
val,0.0196,0.2007,0.5151,0.0196,0.0


PASS  Same feature semantics in train / val / test (NaN only when there is no earlier answer)
[  209.6s] audit tests done


## 5. Evaluation toolkit and baselines (Priority 3)

All models are scored on the **same validation rows** and later the **same test rows** (asserted). Metrics:

| Metric | Meaning |
|---|---|
| Accuracy / Balanced accuracy | share correct overall / average of recall on correct and incorrect answers |
| Precision, Recall, F1 | quality of the "will answer correctly" predictions |
| Precision / Recall (incorrect) | how well "will struggle" is detected — the minority class and the one a tutor cares about |
| ROC-AUC | ability to rank correct above incorrect answers (threshold-free) |
| PR-AUC (correct / incorrect) | precision–recall quality for each class, robust to imbalance |
| Log-loss, Brier, ECE | reliability of the predicted probabilities (what the recommender consumes) |

Baselines: **Majority / base-rate** (always predicts the training majority class with the training base rate as probability) and a **student past-accuracy heuristic** (no learning). BKT outputs a *correctness probability* (`P(L)(1-slip)+(1-P(L))guess`), so it is comparable with the other models; the raw mastery score is a different quantity.

In [10]:
from sklearn.metrics import roc_auc_score, average_precision_score
EPS = 1e-7
idx_tr, idx_val, idx_te = (np.flatnonzero(split == k) for k in (0, 1, 2))
y_tr, y_val, y_te = y_all[idx_tr], y_all[idx_val], y_all[idx_te]
stu_val, stu_te = student_all[idx_val], student_all[idx_te]

def _logloss(y, p):
    pc = np.clip(np.asarray(p, dtype=np.float64), EPS, 1 - EPS); y = np.asarray(y, dtype=np.float64)
    return float(-np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc)))

def ece_score(y, p, bins=20):
    y = np.asarray(y, dtype=np.float64); p = np.asarray(p, dtype=np.float64)
    edges = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
    if len(edges) < 2: return float(abs(p.mean() - y.mean()))
    b = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, len(edges) - 2)
    cnt = np.bincount(b, minlength=len(edges) - 1).astype(float); sp = np.bincount(b, weights=p, minlength=len(edges) - 1)
    sy = np.bincount(b, weights=y, minlength=len(edges) - 1); ok = cnt > 0
    return float(np.sum(np.abs(sp[ok] - sy[ok])) / len(y))

def metrics_at(y, p, thr=0.5):
    y = np.asarray(y).astype(np.int8); p = np.asarray(p, dtype=np.float64); pred = p >= thr
    tp, fp = int(np.sum(pred & (y == 1))), int(np.sum(pred & (y == 0)))
    tn, fn = int(np.sum(~pred & (y == 0))), int(np.sum(~pred & (y == 1)))
    n, P, N = len(y), tp + fn, tn + fp
    prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / P if P else 0.0
    spec = tn / N if N else 0.0; prec_neg = tn / (tn + fn) if tn + fn else 0.0
    return {"threshold": float(thr), "n": n, "accuracy": (tp + tn) / n, "balanced_accuracy": (rec + spec) / 2,
            "precision": prec, "recall": rec, "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
            "precision_incorrect": prec_neg, "recall_incorrect": spec,
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p)),
            "pr_auc_incorrect": float(average_precision_score(1 - y, 1 - p)),
            "log_loss": _logloss(y, p), "brier": float(np.mean((p - y) ** 2)), "ece": ece_score(y, p),
            "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def tune_thresholds(y, p, lo=0.30, hi=0.85, step=0.005):
    """Threshold maximising accuracy / balanced accuracy. Call with VALIDATION data only."""
    ths = np.round(np.arange(lo, hi + 1e-9, step), 4)
    order = np.argsort(p, kind="stable"); ps = p[order]; ys = y[order].astype(np.int64)
    cum = np.concatenate([[0], np.cumsum(ys)]); n = len(y); P = int(cum[-1]); N = n - P
    k = np.searchsorted(ps, ths, side="left"); fn = cum[k]; tn = k - fn; tp = P - fn
    table = pd.DataFrame({"threshold": ths, "accuracy": (tp + tn) / n, "balanced_accuracy": (tp / P + tn / N) / 2,
                          "recall_correct": tp / P, "recall_incorrect": tn / N})
    return float(ths[np.argmax(table["accuracy"])]), float(ths[np.argmax(table["balanced_accuracy"])]), table

def cluster_bootstrap(student, y, p_a, p_b, thr_a=0.5, thr_b=0.5, B=300, seed=0):
    """Student-level bootstrap of (model A - model B) differences. Rows of one student are correlated, so students are resampled."""
    sid, inv = np.unique(student, return_inverse=True); S = len(sid); y = y.astype(np.float64)
    n = np.bincount(inv, minlength=S).astype(float)
    def per_student(p, thr):
        pc = np.clip(p, EPS, 1 - EPS)
        return (np.bincount(inv, weights=((p >= thr) == (y == 1)).astype(float), minlength=S),
                np.bincount(inv, weights=-(y * np.log(pc) + (1 - y) * np.log(1 - pc)), minlength=S))
    ca, la = per_student(p_a, thr_a); cb, lb = per_student(p_b, thr_b)
    rng = np.random.default_rng(seed); da, dl = [], []
    for _ in range(B):
        w = rng.multinomial(S, np.full(S, 1.0 / S)).astype(float); tot = (w * n).sum()
        da.append(((w * ca).sum() - (w * cb).sum()) / tot); dl.append(((w * la).sum() - (w * lb).sum()) / tot)
    q = lambda a: (float(np.mean(a)), float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))
    return {"acc_diff": q(da), "logloss_diff": q(dl), "students": S}

def predict_batched(model, X, bs=500_000):
    return np.concatenate([model.predict_proba(X.iloc[i:i + bs])[:, 1] for i in range(0, len(X), bs)])

PRED, SELECTABLE, NOTES = {}, {}, {}
def register(name, p_val, p_te, selectable, note=""):
    assert len(p_val) == len(idx_val) and len(p_te) == len(idx_te), "every model must be scored on the SAME held-out rows"
    PRED[name] = {"val": np.asarray(p_val, dtype=np.float64), "test": np.asarray(p_te, dtype=np.float64)}
    SELECTABLE[name], NOTES[name] = selectable, note

register("Majority / base-rate", np.full(len(idx_val), PRIOR), np.full(len(idx_te), PRIOR), False, "always predicts train-majority class; probability = train base rate")
heur = feat["stu_acc"].fillna(PRIOR).to_numpy()
register("Student past-accuracy heuristic", heur[idx_val], heur[idx_te], False, "no learning: P(correct)=student's accuracy so far")
print("Train majority class:", int(PRIOR >= 0.5), "| majority accuracy on VAL:", round(max(y_val.mean(), 1 - y_val.mean()), 4),
      "| on TEST:", round(max(y_te.mean(), 1 - y_te.mean()), 4))

Train majority class: 1 | majority accuracy on VAL: 0.68 | on TEST: 0.7071


## 6. Models (Priority 3 / 4)

Every learned model sees the **same strict features** (§3) and the same random sample of training rows (`HGB_TRAIN_ROWS`; sampling rows is safe because the features were already computed from past-only history).

* **Logistic Regression** – log-scaled counts, median-imputed + missing-indicator numerics, one-hot categoricals. Built only from scikit-learn parts, so the saved file loads anywhere.
* **HistGradientBoosting** – config A is the teammate's setting; config B is deeper/more regularised. The better one is chosen on **validation log-loss**.
* **HistGB "loose" (sensitivity only)** – the teammate's non-strict variant. It lets a row see earlier rows of its own timestamp bucket, whose true order is unknown. It is trained only to measure how much this matters and is **not eligible for selection**.
* **BKT** – per student × concept, with strict timestamp semantics (§6.3): fixed guessed parameters, and parameters **fitted on the training period**.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

fit_idx = np.sort(np.random.default_rng(SEED).choice(idx_tr, size=min(HGB_TRAIN_ROWS, len(idx_tr)), replace=False))
X_fit, y_fit = feat.iloc[fit_idx], y_all[fit_idx]
X_val, X_te = feat.iloc[idx_val], feat.iloc[idx_te]
print(f"Fit sample: {len(X_fit):,} train rows | val rows: {len(X_val):,} | test rows: {len(X_te):,}")

# ------------------------- Logistic Regression -------------------------
LOG_COLS = ["stu_attempts", "concept_attempts", "exercise_attempts", "gap_minutes"]
LIN_COLS = ["stu_acc", "prev_correct", "recent10_acc", "concept_acc", "exercise_acc", "exercise_enc", "concept_enc"]
def make_lr():
    num = lambda log: Pipeline(([("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one"))] if log else [])
                               + [("impute", SimpleImputer(strategy="median", add_indicator=True)), ("scale", StandardScaler())])
    prep = ColumnTransformer([("log", num(True), LOG_COLS), ("lin", num(False), LIN_COLS),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES)])
    return Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=300, C=1.0))])

lr_model = make_lr().fit(X_fit, y_fit)
register("Logistic Regression", predict_batched(lr_model, X_val), predict_batched(lr_model, X_te), True,
         "same strict features; imputation + missing indicators")
MODELS = {"Logistic Regression": lr_model}
tick("Logistic Regression done")

# ------------------------- HistGradientBoosting -------------------------
HGB_CONFIGS = {"HistGB-A (63 leaves, lr 0.10)": dict(max_iter=150, learning_rate=0.10, max_leaf_nodes=63)}
if TUNE_HGB:
    HGB_CONFIGS["HistGB-B (127 leaves, lr 0.06, L2)"] = dict(max_iter=300, learning_rate=0.06, max_leaf_nodes=127,
                                                              l2_regularization=1.0, min_samples_leaf=100)
def make_hgb(cfg):
    return HistGradientBoostingClassifier(categorical_features=CAT_FEATURES, early_stopping=True, validation_fraction=0.05,
                                          n_iter_no_change=15, random_state=SEED, **cfg)
for name, cfg in HGB_CONFIGS.items():
    m = make_hgb(cfg).fit(X_fit, y_fit)
    register(name, predict_batched(m, X_val), predict_batched(m, X_te), True, f"strict features; {m.n_iter_} boosting iterations used")
    MODELS[name] = m; tick(f"{name} done ({m.n_iter_} iterations)")

if RUN_LOOSE_SENSITIVITY:
    hist_loose, _ = build_history_features(raw["student"].to_numpy(), raw["concept"].to_numpy(), raw["exercise"].to_numpy().astype(np.int64),
                                           raw["ts"].to_numpy(), raw["y"].to_numpy(), raw["sess"].to_numpy(), raw["pn"].to_numpy(),
                                           raw["rowid"].to_numpy(), n_concepts, n_ex, strict=False)
    feat_loose = feat.copy()
    for k in HISTORY_FEATURES: feat_loose[k] = hist_loose[k]
    del hist_loose
    first_cfg = next(iter(HGB_CONFIGS.values()))
    m_loose = make_hgb(first_cfg).fit(feat_loose.iloc[fit_idx], y_fit)
    register("HistGB-A LOOSE (sensitivity only)", predict_batched(m_loose, feat_loose.iloc[idx_val]), predict_batched(m_loose, feat_loose.iloc[idx_te]),
             False, "rows may see earlier rows of the SAME timestamp bucket (order unknown) -> not selectable")
    del feat_loose, m_loose; gc.collect(); tick("loose sensitivity model done")

Fit sample: 3,000,000 train rows | val rows: 2,432,488 | test rows: 2,432,333
[  237.4s] Logistic Regression done
[  306.0s] HistGB-A (63 leaves, lr 0.10) done (150 iterations)
[  447.1s] HistGB-B (127 leaves, lr 0.06, L2) done (300 iterations)
[  546.2s] loose sensitivity model done


### 6.3 Bayesian Knowledge Tracing (BKT), strict version

State = P(student has mastered the concept). For each `(student, concept)` sequence: predict `P(correct) = P(L)(1−slip) + (1−P(L))·guess`, then update `P(L)` from the observed answer and apply the learning transition.
**Strictness:** all rows of the same `(student, concept, timestamp)` are predicted from the mastery *before* that timestamp (so BKT obeys the same no-peeking rule as the other models). Training rows only update the state; validation/test rows are predicted before they update it.
Parameters are global (not per concept): `fixed` = the earlier guessed values; `fitted` = Nelder–Mead on **training rows only** (log-loss).

In [12]:
def _bkt_py(key, tsx, y, p_l0, p_t, p_g, p_s, pred, m_after):
    m = p_l0; gm = p_l0; prev_key = -1; prev_ts = -1
    for i in range(len(y)):
        if key[i] != prev_key:
            m = p_l0; gm = p_l0; prev_key = key[i]; prev_ts = tsx[i]
        elif tsx[i] != prev_ts:
            gm = m; prev_ts = tsx[i]                           # mastery visible only from strictly earlier timestamps
        pred[i] = gm * (1.0 - p_s) + (1.0 - gm) * p_g
        if y[i] == 1:
            num = m * (1.0 - p_s); den = num + (1.0 - m) * p_g
        else:
            num = m * p_s; den = num + (1.0 - m) * (1.0 - p_g)
        m = num / den
        m = m + (1.0 - m) * p_t
        m_after[i] = m

try:
    from numba import njit
    bkt_kernel = njit(cache=False)(_bkt_py); BKT_ENGINE = "numba"
    _a = np.zeros(2); bkt_kernel(np.array([0, 0], dtype=np.int64), np.array([1, 2], dtype=np.int64), np.array([1, 0], dtype=np.int64), .2, .15, .2, .1, _a, _a.copy())
except Exception:
    bkt_kernel = _bkt_py; BKT_ENGINE = "python (install numba for ~50x speed)"
print("BKT engine:", BKT_ENGINE)

def run_bkt(key_s, ts_s, y_s, params):
    pred = np.empty(len(y_s)); m_after = np.empty(len(y_s))
    bkt_kernel(key_s, ts_s, y_s, float(params[0]), float(params[1]), float(params[2]), float(params[3]), pred, m_after)
    return pred, m_after

BKT_FIXED = (0.20, 0.15, 0.20, 0.10)                      # (P_L0, P_T, P_G, P_S): the earlier guessed values

# ---- BKT label-flip test: an answer must not influence predictions at or before its own timestamp ----
def subset_ctx(y_override):
    return build_history_features(sub["student"], sub["concept"], sub["exercise"].astype(np.int64), sub["ts"], y_override,
                                  sub["sess"], sub["pn"], sub["rowid"], n_concepts, n_ex, strict=True)[1]
def bkt_rows(y_override):
    c = subset_ctx(y_override); p, _ = run_bkt(c["key_s"], c["ts_s"], c["y_s"], BKT_FIXED)
    out = np.empty(len(p)); out[c["order"]] = p; return out
base_p = bkt_rows(sub["y"]); viol = 0; later_changed = 0
for r in np.random.default_rng(3).choice(len(sidx), size=30, replace=False):
    y2 = sub["y"].copy(); y2[r] = 1 - y2[r]; p2 = bkt_rows(y2)
    ck_all = sub["student"].astype(np.int64) * n_concepts + sub["concept"]
    same = ck_all == ck_all[r]
    must_equal = (~same) | (sub["ts"] <= sub["ts"][r])
    viol += int((np.abs(p2 - base_p)[must_equal] > 1e-12).sum()); later_changed += int(((np.abs(p2 - base_p) > 1e-12) & same & (sub["ts"] > sub["ts"][r])).any())
record("BKT label-flip test: an answer never changes BKT predictions at/before its own timestamp", viol == 0,
       f"30 flips, violations={viol}; later rows changed in {later_changed}/30 flips")

# ---- fit BKT parameters on TRAIN rows only ----
from scipy.optimize import minimize
key_s_all, ts_s_all, y_s_all, ord_c = bkt_ctx["key_s"], bkt_ctx["ts_s"], bkt_ctx["y_s"], bkt_ctx["order"]
train_pos = np.flatnonzero(split[ord_c] == 0)                  # train rows form a time-prefix of every (student, concept) sequence
FIT_ROWS = 1_500_000 if BKT_ENGINE == "numba" else 300_000      # pure-python fallback is ~50x slower
if len(train_pos) > FIT_ROWS:
    q = FIT_ROWS / len(train_pos)
    keep_student = np.random.default_rng(SEED).random(int(raw["student"].max()) + 1) < q
    train_pos = train_pos[keep_student[key_s_all[train_pos] // n_concepts]]
k_fit, t_fit, yb_fit = key_s_all[train_pos], ts_s_all[train_pos], y_s_all[train_pos]
obj = lambda th: _logloss(yb_fit, run_bkt(k_fit, t_fit, yb_fit, th)[0])
res = minimize(obj, x0=np.array(BKT_FIXED), method="Nelder-Mead",
               bounds=[(0.01, 0.95), (0.001, 0.60), (0.01, 0.45), (0.01, 0.45)], options=dict(maxiter=120, xatol=1e-3, fatol=1e-6))
BKT_FITTED = tuple(float(v) for v in res.x)
print(f"BKT fit rows: {len(k_fit):,} | evaluations: {res.nfev} | train log-loss fixed {obj(np.array(BKT_FIXED)):.4f} -> fitted {res.fun:.4f}")
print("Fitted (P_L0, P_T, P_G, P_S) =", tuple(round(v, 4) for v in BKT_FITTED))

bkt_final_mastery = None
for tag, params in [("BKT (fixed guessed params)", BKT_FIXED), ("BKT (fitted on train)", BKT_FITTED)]:
    p_sorted, m_after = run_bkt(key_s_all, ts_s_all, y_s_all, params)
    p_rows = np.empty(len(p_sorted)); p_rows[ord_c] = p_sorted
    register(tag, p_rows[idx_val], p_rows[idx_te], False, "strict BKT; predicted before update; global parameters")
    if tag.startswith("BKT (fitted"): bkt_final_mastery = m_after
    del p_sorted, p_rows
tick("BKT done")

BKT engine: numba
PASS  BKT label-flip test: an answer never changes BKT predictions at/before its own timestamp  [30 flips, violations=0; later rows changed in 13/30 flips]
BKT fit rows: 1,471,949 | evaluations: 94 | train log-loss fixed 0.7216 -> fitted 0.5716
Fitted (P_L0, P_T, P_G, P_S) = (0.7664, 0.001, 0.45, 0.1958)
[  553.5s] BKT done


## 7. Model selection and threshold tuning — validation data only

* **Selection rule:** among learned, strict models, the lowest **validation log-loss** wins (a proper scoring rule; the recommender consumes probabilities, so reliable probabilities matter more than accuracy at one cut-off).
* **Thresholds** (applied later to the untouched test set): `accuracy-optimal` and `balanced-accuracy-optimal` (treats "will struggle" and "will succeed" equally — usually the better rule for flagging weak topics). Default 0.5 is always reported too.
* Accuracy must be read **next to the majority baseline of the same rows**: the correct-answer rate is about 68 % in validation and about 71 % in test, so "68 % accuracy" means something different in each period.

In [13]:
maj_val = max(y_val.mean(), 1 - y_val.mean()); maj_te = max(y_te.mean(), 1 - y_te.mean())
val_rows = []
for name, d in PRED.items():
    r = metrics_at(y_val, d["val"], 0.5); r.update(model=name, split="validation", selectable=SELECTABLE[name]); val_rows.append(r)
val_tbl = pd.DataFrame(val_rows).set_index("model")
val_tbl["majority_accuracy_same_rows"] = maj_val; val_tbl["accuracy_gain_vs_majority_pp"] = (val_tbl["accuracy"] - maj_val) * 100
show = ["accuracy", "majority_accuracy_same_rows", "accuracy_gain_vs_majority_pp", "balanced_accuracy", "roc_auc", "pr_auc", "log_loss", "brier", "ece"]
print(f"VALIDATION  ({len(y_val):,} rows, correct-rate {y_val.mean():.4f}, majority accuracy {maj_val:.4f})")
display(val_tbl[show].round(4))

cands = [n for n in PRED if SELECTABLE[n]]
FINAL_NAME = min(cands, key=lambda n: val_tbl.loc[n, "log_loss"])
final_model = MODELS[FINAL_NAME]
print(f"\nSELECTED (lowest validation log-loss among strict learned models): {FINAL_NAME}")
print("Candidates:", {n: round(val_tbl.loc[n, 'log_loss'], 4) for n in cands})

THR = {}
for name, d in PRED.items():
    if name.startswith("Majority"): THR[name] = (0.5, 0.5); continue
    ta, tb, _ = tune_thresholds(y_val, d["val"]); THR[name] = (ta, tb)
t_acc, t_bal, thr_table = tune_thresholds(y_val, PRED[FINAL_NAME]["val"])
print(f"Final model thresholds tuned on validation: accuracy-optimal = {t_acc}, balanced-accuracy-optimal = {t_bal}")
display(thr_table.iloc[::10].round(4))

VALIDATION  (2,432,488 rows, correct-rate 0.6800, majority accuracy 0.6800)


,accuracy,majority_accuracy_same_rows,accuracy_gain_vs_majority_pp,balanced_accuracy,roc_auc,pr_auc,log_loss,brier,ece
model,,,,,,,,,
Majority / base-rate,0.6800,0.68,0.0000,0.5000,0.5000,0.6800,0.6288,0.2184,0.0281
Student past-accuracy heuristic,0.6975,0.68,1.7509,0.5716,0.6731,0.7970,0.6336,0.2012,0.0203
Logistic Regression,0.7205,0.68,4.0504,0.6228,0.7343,0.8457,0.5498,0.1853,0.0057
"HistGB-A (63 leaves, lr 0.10)",0.7241,0.68,4.4160,0.6325,0.7416,0.8516,0.5435,0.1830,0.0070
"HistGB-B (127 leaves, lr 0.06, L2)",0.7238,0.68,4.3844,0.6319,0.7415,0.8517,0.5437,0.1831,0.0065
HistGB-A LOOSE (sensitivity only),0.7349,0.68,5.4982,0.6485,0.7588,0.8618,0.5303,0.1775,0.0035
BKT (fixed guessed params),0.6016,0.68,-7.8390,0.5280,0.5906,0.7461,0.7520,0.2608,0.2048
BKT (fitted on train),0.6992,0.68,1.9243,0.6012,0.6599,0.7826,0.5927,0.2022,0.0339



SELECTED (lowest validation log-loss among strict learned models): HistGB-A (63 leaves, lr 0.10)
Candidates: {'Logistic Regression': np.float64(0.5498), 'HistGB-A (63 leaves, lr 0.10)': np.float64(0.5435), 'HistGB-B (127 leaves, lr 0.06, L2)': np.float64(0.5437)}
Final model thresholds tuned on validation: accuracy-optimal = 0.49, balanced-accuracy-optimal = 0.665


,threshold,accuracy,balanced_accuracy,recall_correct,recall_incorrect
0,0.30,0.7003,0.5399,0.9854,0.0945
10,0.35,0.7091,0.5610,0.9725,0.1495
20,0.40,0.7172,0.5847,0.9528,0.2165
30,0.45,0.7229,0.6090,0.9255,0.2926
40,0.50,0.7241,0.6325,0.8871,0.3778
50,0.55,0.7195,0.6528,0.8381,0.4675
60,0.60,0.7075,0.6679,0.7780,0.5578
70,0.65,0.6857,0.6758,0.7033,0.6483
80,0.70,0.6520,0.6739,0.6130,0.7348
90,0.75,0.6071,0.6623,0.5089,0.8157


## 8. Final test evaluation — scored once, same rows for every model

Test rows were not used for fitting, feature fitting, model selection, or threshold tuning.

In [14]:
test_rows = []
for name, d in PRED.items():
    for kind, thr in [("default_0.5", 0.5), ("accuracy_opt(val)", THR[name][0]), ("balanced_opt(val)", THR[name][1])]:
        r = metrics_at(y_te, d["test"], thr); r.update(model=name, split="test", threshold_kind=kind, selectable=SELECTABLE[name]); test_rows.append(r)
test_long = pd.DataFrame(test_rows)
test_long["majority_accuracy_same_rows"] = maj_te
test_long["accuracy_gain_vs_majority_pp"] = (test_long["accuracy"] - maj_te) * 100
test_tbl = test_long[test_long.threshold_kind == "default_0.5"].set_index("model")

print(f"TEST  ({len(y_te):,} rows, correct-rate {y_te.mean():.4f}, majority accuracy {maj_te:.4f})  -- threshold 0.5 for every model")
display(test_tbl[show].round(4))
print(f"FINAL MODEL = {FINAL_NAME}: test metrics at the validation-tuned thresholds")
display(test_long[(test_long.model == FINAL_NAME)].set_index("threshold_kind")[
    ["threshold", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "precision_incorrect", "recall_incorrect", "roc_auc", "pr_auc", "pr_auc_incorrect", "log_loss"]].round(4))

fp_te = PRED[FINAL_NAME]["test"]; tn_, fp_, fn_, tp_ = (test_long[(test_long.model == FINAL_NAME) & (test_long.threshold_kind == "balanced_opt(val)")][["tn", "fp", "fn", "tp"]].iloc[0])
print("Confusion matrix at balanced-optimal threshold  [rows=actual incorrect/correct, cols=pred incorrect/correct]:")
print(np.array([[tn_, fp_], [fn_, tp_]]))
print("\nTemporal shift check: correct-rate validation =", round(y_val.mean(), 4), "| test =", round(y_te.mean(), 4),
      "| majority accuracy validation =", round(maj_val, 4), "| test =", round(maj_te, 4))

TEST  (2,432,333 rows, correct-rate 0.7071, majority accuracy 0.7071)  -- threshold 0.5 for every model


,accuracy,majority_accuracy_same_rows,accuracy_gain_vs_majority_pp,balanced_accuracy,roc_auc,pr_auc,log_loss,brier,ece
model,,,,,,,,,
Majority / base-rate,0.7071,0.7071,0.0000,0.5000,0.5000,0.7071,0.6047,0.2071,0.0010
Student past-accuracy heuristic,0.7218,0.7071,1.4679,0.5698,0.6755,0.8151,0.6226,0.1913,0.0227
Logistic Regression,0.7389,0.7071,3.1744,0.6097,0.7336,0.8592,0.5312,0.1772,0.0064
"HistGB-A (63 leaves, lr 0.10)",0.7416,0.7071,3.4440,0.6190,0.7421,0.8658,0.5244,0.1748,0.0045
"HistGB-B (127 leaves, lr 0.06, L2)",0.7413,0.7071,3.4217,0.6183,0.7421,0.8658,0.5244,0.1748,0.0046
HistGB-A LOOSE (sensitivity only),0.7509,0.7071,4.3730,0.6352,0.7597,0.8754,0.5113,0.1695,0.0020
BKT (fixed guessed params),0.6185,0.7071,-8.8661,0.5292,0.5911,0.7669,0.7200,0.2486,0.1929
BKT (fitted on train),0.7219,0.7071,1.4803,0.5983,0.6588,0.8012,0.5700,0.1919,0.0289


FINAL MODEL = HistGB-A (63 leaves, lr 0.10): test metrics at the validation-tuned thresholds


,threshold,accuracy,balanced_accuracy,precision,recall,f1,precision_incorrect,recall_incorrect,roc_auc,pr_auc,pr_auc_incorrect,log_loss
threshold_kind,,,,,,,,,,,,
default_0.5,0.500,0.7416,0.6190,0.7654,0.9149,0.8335,0.6113,0.3230,0.7421,0.8658,0.5465,0.5244
accuracy_opt(val),0.490,0.7416,0.6144,0.7626,0.9216,0.8346,0.6187,0.3071,0.7421,0.8658,0.5465,0.5244
balanced_opt(val),0.665,0.7003,0.6734,0.8199,0.7383,0.7770,0.4906,0.6084,0.7421,0.8658,0.5465,0.5244


Confusion matrix at balanced-optimal threshold  [rows=actual incorrect/correct, cols=pred incorrect/correct]:
[[ 433384  278982]
 [ 450042 1269925]]

Temporal shift check: correct-rate validation = 0.68 | test = 0.7071 | majority accuracy validation = 0.68 | test = 0.7071


In [15]:
# ---- paired, student-level bootstrap: is the final model really better, and by how much? ----
ref_names = ["Majority / base-rate", "Student past-accuracy heuristic", "BKT (fitted on train)", "BKT (fixed guessed params)", "Logistic Regression"]
boot_rows = []
for ref in [r for r in ref_names if r in PRED and r != FINAL_NAME]:
    b = cluster_bootstrap(stu_te, y_te, PRED[FINAL_NAME]["test"], PRED[ref]["test"], 0.5, 0.5, B=BOOTSTRAP_B, seed=SEED)
    boot_rows.append({"final_model": FINAL_NAME, "vs": ref, "students_resampled": b["students"],
                      "accuracy_diff_pp": 100 * b["acc_diff"][0], "acc_ci95_low_pp": 100 * b["acc_diff"][1], "acc_ci95_high_pp": 100 * b["acc_diff"][2],
                      "logloss_diff": b["logloss_diff"][0], "ll_ci95_low": b["logloss_diff"][1], "ll_ci95_high": b["logloss_diff"][2]})
boot_tbl = pd.DataFrame(boot_rows); print("Final model minus reference (positive accuracy diff / NEGATIVE log-loss diff = final model better); 95% student-bootstrap CI")
display(boot_tbl.round(4))

# ---- subgroup behaviour of the final model on test ----
def safe_metrics(y, p, thr=0.5):
    if len(y) < 50 or len(np.unique(y)) < 2: return {}
    m = metrics_at(y, p, thr); return {k: m[k] for k in ["accuracy", "balanced_accuracy", "roc_auc", "log_loss", "ece"]}
att_te = feat["stu_attempts"].to_numpy()[idx_te]
unseen = ~np.isin(stu_te, list(train_students))
groups = {"all test rows": np.ones(len(idx_te), bool),
          "students seen in training": ~unseen, "students NEVER seen in training": unseen,
          "first 5 answers of the student": att_te < 5, "5-49 earlier answers": (att_te >= 5) & (att_te < 50),
          "50-499 earlier answers": (att_te >= 50) & (att_te < 500), "500+ earlier answers": att_te >= 500}
dcode = feat["difficulty_code"].to_numpy()[idx_te]
for i, lv in enumerate(difficulty_levels): groups[f"difficulty = {lv}"] = dcode == i
groups["difficulty missing"] = np.isnan(dcode)
sg = []
for g, m in groups.items():
    if m.sum() == 0: continue
    mm = safe_metrics(y_te[m], fp_te[m]); 
    if not mm: continue
    sg.append({"group": g, "rows": int(m.sum()), "share": m.mean(), "correct_rate": y_te[m].mean(),
               "majority_accuracy": max(y_te[m].mean(), 1 - y_te[m].mean()), **mm,
               "heuristic_accuracy": safe_metrics(y_te[m], PRED["Student past-accuracy heuristic"]["test"][m]).get("accuracy", np.nan)})
subgroup_tbl = pd.DataFrame(sg); display(subgroup_tbl.round(4))

Final model minus reference (positive accuracy diff / NEGATIVE log-loss diff = final model better); 95% student-bootstrap CI


,final_model,vs,students_resampled,accuracy_diff_pp,acc_ci95_low_pp,acc_ci95_high_pp,logloss_diff,ll_ci95_low,ll_ci95_high
0,"HistGB-A (63 leaves, lr 0.10)",Majority / base-rate,23270,3.4515,3.1852,3.7227,-0.0804,-0.0830,-0.0781
1,"HistGB-A (63 leaves, lr 0.10)",Student past-accuracy heuristic,23270,1.9766,1.8434,2.1026,-0.0982,-0.1012,-0.0951
2,"HistGB-A (63 leaves, lr 0.10)",BKT (fitted on train),23270,1.9633,1.8746,2.0472,-0.0457,-0.0466,-0.0446
3,"HistGB-A (63 leaves, lr 0.10)",BKT (fixed guessed params),23270,12.3127,12.0920,12.5349,-0.1956,-0.1980,-0.1932
4,"HistGB-A (63 leaves, lr 0.10)",Logistic Regression,23270,0.2698,0.2234,0.3172,-0.0069,-0.0072,-0.0065


,group,rows,share,correct_rate,majority_accuracy,accuracy,balanced_accuracy,roc_auc,log_loss,ece,heuristic_accuracy
0,all test rows,2432333,1.0000,0.7071,0.7071,0.7416,0.6190,0.7421,0.5244,0.0045,0.7218
1,students seen in training,1152950,0.4740,0.7038,0.7038,0.7368,0.6203,0.7375,0.5305,0.0087,0.7170
2,students NEVER seen in training,1279383,0.5260,0.7101,0.7101,0.7458,0.6176,0.7464,0.5188,0.0031,0.7261
3,first 5 answers of the student,93449,0.0384,0.7701,0.7701,0.7699,0.5575,0.7050,0.4924,0.0176,0.7554
4,5-49 earlier answers,279133,0.1148,0.7423,0.7423,0.7617,0.5888,0.7325,0.5020,0.0060,0.7489
5,50-499 earlier answers,1089624,0.4480,0.7053,0.7053,0.7417,0.6218,0.7449,0.5236,0.0048,0.7230
6,500+ earlier answers,970127,0.3988,0.6930,0.6930,0.7328,0.6260,0.7420,0.5347,0.0046,0.7095
7,difficulty = easy,1632771,0.6713,0.7419,0.7419,0.7663,0.6043,0.7447,0.4944,0.0053,0.7508
8,difficulty = hard,236127,0.0971,0.6266,0.6266,0.6893,0.6360,0.7200,0.5872,0.0060,0.6538
9,difficulty = normal,497761,0.2046,0.6304,0.6304,0.6827,0.6229,0.7060,0.5950,0.0070,0.6574


In [16]:
# ---- figures: ROC / PR, reliability, feature importance ----
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
def thin(x, y, k=2000):
    i = np.unique(np.linspace(0, len(x) - 1, k).astype(int)); return x[i], y[i]
plot_models = [FINAL_NAME, "Logistic Regression", "BKT (fitted on train)", "Student past-accuracy heuristic"]
plot_models = list(dict.fromkeys(n for n in plot_models if n in PRED))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for n in plot_models:
    f, t, _ = roc_curve(y_te, PRED[n]["test"]); ax[0].plot(*thin(f, t), label=f"{n} (AUC {roc_auc_score(y_te, PRED[n]['test']):.3f})")
    pr, rc, _ = precision_recall_curve(y_te, PRED[n]["test"]); ax[1].plot(*thin(rc, pr), label=n)
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set(title="ROC (test)", xlabel="False-positive rate", ylabel="True-positive rate"); ax[0].legend(fontsize=7)
ax[1].axhline(y_te.mean(), color="k", ls="--", lw=.8); ax[1].set(title="Precision-Recall, class 'correct' (test)", xlabel="Recall", ylabel="Precision")
plt.tight_layout(); plt.savefig(WORK / "member2_roc_pr.png", dpi=130); plt.show()

edges = np.unique(np.quantile(fp_te, np.linspace(0, 1, 21))); b = np.clip(np.searchsorted(edges, fp_te, side="right") - 1, 0, len(edges) - 2)
cal = pd.DataFrame({"p": fp_te, "y": y_te, "b": b}).groupby("b").mean()
fig, ax = plt.subplots(figsize=(4.8, 4.6)); ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.plot(cal["p"], cal["y"], "o-")
ax.set(title=f"Reliability (test) - {FINAL_NAME}\nECE={ece_score(y_te, fp_te):.4f}", xlabel="Mean predicted P(correct)", ylabel="Observed correct rate")
plt.tight_layout(); plt.savefig(WORK / "member2_calibration.png", dpi=130); plt.show()

vs = np.random.default_rng(SEED).choice(len(idx_val), size=min(100_000, len(idx_val)), replace=False)
pi = permutation_importance(final_model, X_val.iloc[vs], y_val[vs], scoring="neg_log_loss", n_repeats=3, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(6.5, 4.6)); imp.plot.barh(ax=ax); ax.set(title="Permutation importance (validation, increase in log-loss)", xlabel="log-loss increase when shuffled")
plt.tight_layout(); plt.savefig(WORK / "member2_feature_importance.png", dpi=130); plt.show()
importance_tbl = imp.sort_values(ascending=False).round(5).reset_index(); importance_tbl.columns = ["feature", "logloss_increase"]
display(importance_tbl)

,feature,logloss_increase
0,exercise_enc,0.06440
1,stu_acc,0.02919
2,concept_acc,0.00729
3,exercise_acc,0.00544
4,concept_cat,0.00487
5,exercise_attempts,0.00317
6,stu_attempts,0.00314
7,recent10_acc,0.00192
8,concept_attempts,0.00081
9,difficulty_code,0.00037


## 9. Save the final artifacts (Priority 5)

| File | Purpose |
|---|---|
| `member2_final_model.joblib` | selected scikit-learn model (only sklearn/numpy objects → loads without this notebook) |
| `member2_model_card.json` | feature list, thresholds, priors, BKT parameters, timestamp resolution, split dates, metrics, library versions |
| `member2_exercise_lookup.csv` | exercise → concept / difficulty / stage / target-encoded difficulty (all exercises in `Info_Content.csv`) |
| `learning_predictor.py` | online interface: `observe()`, `predict_proba()`, `predict_many()`, `bkt_mastery()` … |
| `bkt_mastery_estimates.csv.gz` | final BKT mastery per (student, concept) |
| `member2_*_metrics.csv`, figures, `member2_final_report.md` | evidence for the report |

In [17]:
%%writefile learning_predictor.py
"""
learning_predictor.py  --  Member 2 handoff module (inference + student-state tracking)

Predicts P(student answers the NEXT exercise correctly) and tracks BKT mastery.
The feature logic here is the online twin of the offline "strict" feature builder in
member2_final_pipeline.ipynb; the notebook verifies the two agree (parity test).

Typical use (Member 3):

    from learning_predictor import LearningPredictor
    lp = LearningPredictor.from_files("member2_final_model.joblib",
                                      "member2_model_card.json",
                                      "member2_exercise_lookup.csv")

    lp.observe("student_42", "ex_abc", is_correct=True, ts="2026-01-05 10:03:00")   # after every answer
    p  = lp.predict_proba("student_42", "ex_xyz", ts="2026-01-05 10:40:00")           # before answering
    ps = lp.predict_many("student_42", ["ex_1", "ex_2", "ex_3"])                      # candidate exercises
    m  = lp.bkt_mastery("student_42", "ex_xyz")                                       # concept mastery

Design rules (identical to training):
  * Only answers from STRICTLY EARLIER timestamp buckets are visible to a prediction.
    Answers observed inside the current bucket stay "pending" until time moves forward.
    Calling predict with ts=None means "time has moved on" and releases pending answers.
  * Unknown exercise / missing metadata never raises: category codes become NaN/"other"
    and the target-encoded difficulty falls back to the training prior.
  * A student with no history gets NaN history features (the model was trained with
    exactly this cold-start representation).
"""
from __future__ import annotations

import json
from collections import deque

import joblib
import numpy as np
import pandas as pd

__all__ = ["LearningPredictor"]

_NAN = float("nan")


class _StudentState:
    __slots__ = ("attempts", "correct", "last10", "last_y", "last_ts",
                 "pending", "pending_ts", "concept", "exercise", "mastery")

    def __init__(self):
        self.attempts = 0
        self.correct = 0
        self.last10 = deque(maxlen=10)
        self.last_y = None
        self.last_ts = None
        self.pending = []          # [(ucid, concept_key, y)] observed inside the current bucket
        self.pending_ts = None
        self.concept = {}          # concept_key -> [attempts, correct]   (committed only)
        self.exercise = {}         # ucid        -> [attempts, correct]   (committed only)
        self.mastery = {}          # concept_key -> BKT P(L) after committed answers


class LearningPredictor:
    def __init__(self, model, card: dict, lookup: pd.DataFrame):
        self.model = model
        self.card = card
        self.feature_names = list(card["features"])
        self.bucket = card.get("timestamp_bucket_seconds") or None
        self.prior = float(card["priors"]["encoding_prior"])
        self.concept_other = float(card["concept_other_code"])
        self.bkt = {k: float(v) for k, v in card["bkt_params"].items()}
        self.thresholds = {"default": 0.5,
                           "accuracy": float(card["threshold_accuracy"]),
                           "balanced": float(card["threshold_balanced"])}
        lk = lookup.copy()
        lk["ucid"] = lk["ucid"].astype(str)
        self._ex = {
            r[0]: (int(r[1]), float(r[2]), float(r[3]), float(r[4]), float(r[5]), float(r[6]))
            for r in zip(lk["ucid"], lk["concept_code"], lk["concept_cat"], lk["difficulty_code"],
                         lk["stage_code"], lk["exercise_enc"], lk["concept_enc"])
        }
        self._students = {}

    # ------------------------------------------------------------------ loading
    @classmethod
    def from_files(cls, model_path, card_path, lookup_path):
        model = joblib.load(model_path)
        with open(card_path, "r", encoding="utf-8") as fh:
            card = json.load(fh)
        lookup = pd.read_csv(lookup_path, dtype={"ucid": str})
        return cls(model, card, lookup)

    # ------------------------------------------------------------------ helpers
    def _bucket_ts(self, ts):
        """-> epoch seconds floored to the training timestamp resolution, or None."""
        if ts is None:
            return None
        if isinstance(ts, (int, float, np.integer, np.floating)):
            t = pd.Timestamp(float(ts), unit="s", tz="UTC")
        else:
            t = pd.Timestamp(ts)
            t = t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")
        sec = int(t.timestamp())
        if self.bucket:
            sec -= sec % int(self.bucket)
        return sec

    def _state(self, student_id):
        key = str(student_id)
        st = self._students.get(key)
        if st is None:
            st = self._students[key] = _StudentState()
        return st

    def _concept_key(self, ucid):
        ex = self._ex.get(str(ucid))
        return ex[0] if ex is not None else ("unk", str(ucid))

    def _bkt_step(self, m, y):
        pl0, pt, pg, ps = self.bkt["p_l0"], self.bkt["p_t"], self.bkt["p_g"], self.bkt["p_s"]
        if y:
            num = m * (1.0 - ps)
            den = num + (1.0 - m) * pg
        else:
            num = m * ps
            den = num + (1.0 - m) * (1.0 - pg)
        m = num / den
        return m + (1.0 - m) * pt

    def _commit(self, st):
        for ucid, ckey, y in st.pending:
            st.attempts += 1
            st.correct += y
            st.last10.append(y)
            st.last_y = y
            c = st.concept.setdefault(ckey, [0, 0]); c[0] += 1; c[1] += y
            e = st.exercise.setdefault(ucid, [0, 0]); e[0] += 1; e[1] += y
            st.mastery[ckey] = self._bkt_step(st.mastery.get(ckey, self.bkt["p_l0"]), y)
        if st.pending:
            st.last_ts = st.pending_ts
        st.pending = []
        st.pending_ts = None

    def _release(self, st, tsb):
        """Make answers from strictly earlier buckets visible."""
        if st.pending and (tsb is None or tsb > st.pending_ts):
            self._commit(st)

    # ------------------------------------------------------------------ public API
    def observe(self, student_id, ucid, is_correct, ts=None):
        """Record an answer. Timestamps must be non-decreasing per student."""
        st = self._state(student_id)
        tsb = self._bucket_ts(ts) if ts is not None else self._bucket_ts(pd.Timestamp.now("UTC"))
        if st.pending and tsb < st.pending_ts:
            raise ValueError("observe(): timestamps must be non-decreasing per student")
        if st.last_ts is not None and tsb < st.last_ts:
            raise ValueError("observe(): timestamp is older than already committed history")
        self._release(st, tsb)
        ucid = str(ucid)
        st.pending.append((ucid, self._concept_key(ucid), int(bool(is_correct))))
        st.pending_ts = tsb

    def load_history(self, student_id, rows):
        """rows: iterable of (ts, ucid, is_correct) in chronological order."""
        for ts, ucid, y in rows:
            self.observe(student_id, ucid, y, ts)

    def features(self, student_id, ucid, ts=None) -> dict:
        st = self._state(student_id)
        tsb = self._bucket_ts(ts)
        self._release(st, tsb)
        return self._row(st, str(ucid), tsb)

    def _row(self, st, ucid, tsb):
        ex = self._ex.get(ucid)
        if ex is None:
            ckey, ccat, dcode, scode, e_enc, c_enc = ("unk", ucid), self.concept_other, _NAN, _NAN, self.prior, self.prior
        else:
            ckey, ccat, dcode, scode, e_enc, c_enc = ex
        n = st.attempts
        c = st.concept.get(ckey, (0, 0))
        e = st.exercise.get(ucid, (0, 0))
        last10 = st.last10
        gap = _NAN
        if tsb is not None and st.last_ts is not None:
            gap = (tsb - st.last_ts) / 60.0
        return {
            "stu_attempts": float(n),
            "stu_acc": st.correct / n if n else _NAN,
            "prev_correct": float(st.last_y) if n else _NAN,
            "recent10_acc": (sum(last10) / len(last10)) if len(last10) else _NAN,
            "concept_attempts": float(c[0]),
            "concept_acc": c[1] / c[0] if c[0] else _NAN,
            "exercise_attempts": float(e[0]),
            "exercise_acc": e[1] / e[0] if e[0] else _NAN,
            "gap_minutes": gap,
            "exercise_enc": e_enc,
            "concept_enc": c_enc,
            "difficulty_code": dcode,
            "stage_code": scode,
            "concept_cat": ccat,
        }

    def _frame(self, rows):
        return pd.DataFrame(rows, columns=self.feature_names).astype("float32")

    def predict_proba(self, student_id, ucid, ts=None) -> float:
        return float(self.predict_many(student_id, [ucid], ts)[0])

    def predict_many(self, student_id, ucids, ts=None) -> np.ndarray:
        st = self._state(student_id)
        tsb = self._bucket_ts(ts)
        self._release(st, tsb)
        rows = [self._row(st, str(u), tsb) for u in ucids]
        if not rows:
            return np.empty(0)
        return self.model.predict_proba(self._frame(rows))[:, 1]

    def predict_label(self, student_id, ucid, ts=None, mode="balanced") -> int:
        """1 = predicted correct. mode: 'balanced' (default), 'accuracy' or 'default' (0.5)."""
        return int(self.predict_proba(student_id, ucid, ts) >= self.thresholds[mode])

    def rank_by_target(self, student_id, ucids, target=0.7, ts=None):
        """Sort candidates by closeness of predicted P(correct) to `target` (challenging-but-doable)."""
        ucids = list(ucids)
        p = self.predict_many(student_id, ucids, ts)
        order = np.argsort(np.abs(p - target), kind="stable")
        return [(ucids[i], float(p[i])) for i in order]

    def bkt_mastery(self, student_id, ucid) -> float:
        """Current BKT P(mastery) of the concept behind `ucid`, including answers still pending."""
        st = self._state(student_id)
        ckey = self._concept_key(ucid)
        m = st.mastery.get(ckey, self.bkt["p_l0"])
        for _, k, y in st.pending:
            if k == ckey:
                m = self._bkt_step(m, y)
        return float(m)

    def state_summary(self, student_id) -> dict:
        st = self._state(student_id)
        n = st.attempts + len(st.pending)
        c = st.correct + sum(y for _, _, y in st.pending)
        return {"answers": n, "accuracy": (c / n) if n else None, "concepts_seen": len(st.concept)}

Writing learning_predictor.py


In [18]:
import platform as _pf
from importlib import metadata as _md
def _ver(p):
    try: return _md.version(p)
    except Exception: return None
VERSIONS = {p: _ver(p) for p in ["numpy", "pandas", "scikit-learn", "scipy", "joblib", "matplotlib", "numba"]}
VERSIONS["python"] = _pf.python_version()

MODEL_PATH, CARD_PATH, LOOKUP_PATH = WORK / "member2_final_model.joblib", WORK / "member2_model_card.json", WORK / "member2_exercise_lookup.csv"
joblib.dump(final_model, MODEL_PATH, compress=3)
ex_lookup.to_csv(LOOKUP_PATH, index=False)

fm = test_long[test_long.model == FINAL_NAME].set_index("threshold_kind")
keep = ["threshold", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "pr_auc_incorrect", "log_loss", "brier", "ece"]
card = {
    "model_name": FINAL_NAME, "model_class": type(final_model).__name__, "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "data_source": DATA_SOURCE, "target": "is_correct of the NEXT interaction (1 = correct)",
    "features": FEATURES, "categorical_features": CAT_FEATURES, "history_features_are_strict": True,
    "timestamp_bucket_seconds": bucket_seconds, "concept_other_code": MAX_CAT,
    "difficulty_levels": difficulty_levels, "learning_stage_levels": stage_levels,
    "threshold_accuracy": t_acc, "threshold_balanced": t_bal, "threshold_default": 0.5,
    "priors": {"encoding_prior": PRIOR, "encoding_smoothing": SMOOTH_M, "encoding_folds": N_FOLDS},
    "bkt_params": dict(zip(["p_l0", "p_t", "p_g", "p_s"], BKT_FITTED)), "bkt_params_fixed_for_reference": dict(zip(["p_l0", "p_t", "p_g", "p_s"], BKT_FIXED)),
    "split": {"train_end_utc": fmt(t70), "val_end_utc": fmt(t85), "train_rows": int(len(idx_tr)), "val_rows": int(len(idx_val)), "test_rows": int(len(idx_te)),
              "fit_sample_rows": int(len(fit_idx))},
    "excluded_columns_known_only_after_answer": POST_ANSWER_COLUMNS,
    "validation_majority_accuracy": float(maj_val), "test_majority_accuracy": float(maj_te),
    "metrics_test": {k: {m: float(fm.loc[k, m]) for m in keep} for k in fm.index},
    "library_versions": VERSIONS, "seed": SEED,
    "notes": ["P(correct) is computed from strictly earlier timestamp buckets only.", "Pass ts= for best accuracy; ts=None releases all pending answers.",
              "Unknown exercise / missing metadata fall back to NaN codes and the training prior."]}
with open(CARD_PATH, "w", encoding="utf-8") as fh: json.dump(card, fh, indent=2, default=float)

# ---------------- metrics / evidence files ----------------
val_tbl.reset_index().to_csv(WORK / "member2_validation_metrics.csv", index=False)
test_long.drop(columns=["selectable"]).assign(data_source=DATA_SOURCE).to_csv(WORK / "member2_final_metrics.csv", index=False)
boot_tbl.to_csv(WORK / "member2_bootstrap_ci.csv", index=False); subgroup_tbl.to_csv(WORK / "member2_subgroup_metrics.csv", index=False)
thr_table.to_csv(WORK / "member2_threshold_table.csv", index=False); split_summary.to_csv(WORK / "member2_split_summary.csv", index=False)
importance_tbl.to_csv(WORK / "member2_feature_importance.csv", index=False)
json.dump({k: v for k, v in AUDIT.items()}, open(WORK / "member2_audit_results.json", "w"), indent=2)

# ---------------- BKT mastery per (student, concept) at the end of the data ----------------
ks = bkt_ctx["key_s"]; nk = len(ks)
last = np.r_[np.flatnonzero(ks[1:] != ks[:-1]), nk - 1]; first_pos = np.r_[0, last[:-1] + 1]
mastery_tbl = pd.DataFrame({"uuid": uuid_values[ks[last] // n_concepts], "concept_code": (ks[last] % n_concepts).astype(int),
                            "concept_label": concept_labels[ks[last] % n_concepts], "mastery_probability": bkt_final_mastery[last].round(6),
                            "attempts": (last - first_pos + 1).astype(int)})
mastery_tbl.to_csv(WORK / "bkt_mastery_estimates.csv.gz", index=False)
print("Saved:", MODEL_PATH.name, f"({MODEL_PATH.stat().st_size/1e6:.1f} MB)", "|", CARD_PATH.name, "|", LOOKUP_PATH.name, f"({len(ex_lookup):,} exercises)",
      "| bkt_mastery_estimates.csv.gz", f"({len(mastery_tbl):,} student-concept pairs)")

Saved: member2_final_model.joblib (0.6 MB) | member2_model_card.json | member2_exercise_lookup.csv (1,330 exercises) | bkt_mastery_estimates.csv.gz (484,455 student-concept pairs)


## 10. Reload and prediction tests, and training/serving parity

1. Reload the saved model from disk and check it reproduces the in-memory predictions.
2. Exercise the prediction interface on awkward inputs: brand-new student, unknown exercise id, exercise with missing metadata, `ts=None`, batch scoring, bad timestamp order.
3. **Parity test:** replay real students' full histories through the *online* `LearningPredictor` and compare, row by row, with the *offline* feature matrix and the offline model probabilities (and BKT mastery). This proves what Member 3 will serve is what was evaluated.

In [19]:
reloaded = joblib.load(MODEL_PATH)
chk = X_te.iloc[:2000]
d_reload = float(np.max(np.abs(reloaded.predict_proba(chk)[:, 1] - final_model.predict_proba(chk)[:, 1])))
record("Saved model reloads and reproduces in-memory predictions", d_reload < 1e-12, f"max |diff| = {d_reload:.1e}")

import learning_predictor as lpmod
importlib.reload(lpmod)
predictor = lpmod.LearningPredictor.from_files(MODEL_PATH, CARD_PATH, LOOKUP_PATH)

ok, notes = True, []
def check(cond, msg):
    global ok
    ok &= bool(cond); notes.append(("ok  " if cond else "FAIL") + " " + msg)

known_ucid = str(ex_lookup["ucid"].iloc[0]); t0_ = "2019-03-04 10:00:00"
p = predictor.predict_proba("brand_new_student", known_ucid, t0_);                       check(0 < p < 1, f"brand-new student, known exercise -> {p:.3f}")
p = predictor.predict_proba("brand_new_student", "exercise-that-does-not-exist", t0_);   check(0 < p < 1, f"unknown exercise id -> {p:.3f}")
miss = ex_lookup[ex_lookup["difficulty_code"].isna() | ex_lookup["stage_code"].isna()]
if len(miss): p = predictor.predict_proba("brand_new_student", str(miss["ucid"].iloc[0]), t0_); check(0 < p < 1, f"exercise with missing difficulty/stage -> {p:.3f}")
else: notes.append("skip (no exercise with missing metadata in this data)")
p = predictor.predict_proba("brand_new_student", known_ucid);                            check(0 < p < 1, f"ts=None -> {p:.3f}")
for i, (u, y_) in enumerate([(known_ucid, 1), (known_ucid, 0), (str(ex_lookup['ucid'].iloc[1]), True)]):
    predictor.observe("s_demo", u, y_, f"2019-03-04 1{i}:00:00")
cands = [str(u) for u in ex_lookup["ucid"].iloc[:60]] + ["unknown-exercise"]
ps = predictor.predict_many("s_demo", cands, "2019-03-05 09:00:00");                    check(ps.shape == (61,) and np.all((ps > 0) & (ps < 1)), "batch of 61 candidates incl. unknown exercise")
rk = predictor.rank_by_target("s_demo", cands, 0.7, "2019-03-05 09:00:00");              check(len(rk) == 61 and abs(rk[0][1] - 0.7) <= abs(rk[-1][1] - 0.7), "rank_by_target orders by closeness to target")
check(predictor.predict_many("s_demo", [], None).size == 0, "empty candidate list")
check(predictor.predict_label("s_demo", known_ucid, "2019-03-05 09:00:00") in (0, 1), "predict_label returns 0/1")
m_ = predictor.bkt_mastery("s_demo", known_ucid);                                         check(0 < m_ < 1, f"bkt_mastery -> {m_:.3f}")
try: predictor.observe("s_demo", known_ucid, 1, "2000-01-01 00:00:00"); check(False, "out-of-order observe should raise")
except ValueError: check(True, "out-of-order observe raises ValueError")
print("\n".join(notes)); record("Prediction interface handles cold-start, unknown exercise, missing metadata, ts=None, batches, bad order", ok)

PASS  Saved model reloads and reproduces in-memory predictions  [max |diff| = 0.0e+00]
ok   brand-new student, known exercise -> 0.504
ok   unknown exercise id -> 0.695
skip (no exercise with missing metadata in this data)
ok   ts=None -> 0.504
ok   batch of 61 candidates incl. unknown exercise
ok   rank_by_target orders by closeness to target
ok   empty candidate list
ok   predict_label returns 0/1
ok   bkt_mastery -> 0.677
ok   out-of-order observe raises ValueError
PASS  Prediction interface handles cold-start, unknown exercise, missing metadata, ts=None, batches, bad order


In [20]:
# ---------------- online (predictor) vs offline (notebook) parity ----------------
ucid_str = np.asarray(ex_table["ucid"].astype(str))
raw_ts, raw_sess, raw_pn, raw_rid, raw_ex = (raw[c].to_numpy() for c in ["ts", "sess", "pn", "rowid", "exercise"])
rows_per_student = np.bincount(student_all)
test_students = np.unique(stu_te)
cand_students = test_students[rows_per_student[test_students] <= 600]
if len(cand_students) < 10: cand_students = test_students[np.argsort(rows_per_student[test_students])[:25]]
par_students = np.random.default_rng(21).choice(cand_students, size=min(40, len(cand_students)), replace=False)

pos_in_te = {int(r): i for i, r in enumerate(idx_te)}
parity = predictor.__class__.from_files(MODEL_PATH, CARD_PATH, LOOKUP_PATH)      # fresh state
max_hist = max_enc = max_p = 0.0; n_rows = n_te_rows = bad_nan = 0
mast_off = mastery_tbl.set_index(["uuid", "concept_code"])["mastery_probability"]; max_bkt = 0.0
hist_cols = [FEATURES.index(c) for c in HISTORY_FEATURES]; enc_cols = [FEATURES.index(c) for c in ENC_FEATURES + CAT_FEATURES]
for s in par_students:
    rows = np.flatnonzero(student_all == s); rows = rows[np.lexsort((raw_rid[rows], raw_pn[rows], raw_sess[rows], raw_ts[rows]))]
    for r in rows:
        f_on = parity.features(f"s{s}", ucid_str[raw_ex[r]], float(raw_ts[r]))
        on = np.array([f_on[c] for c in FEATURES], dtype=np.float64); off = feat.iloc[r].to_numpy(dtype=np.float64)
        bad_nan += int(np.any(np.isnan(on[hist_cols]) != np.isnan(off[hist_cols])))
        d = np.nan_to_num(np.abs(on[hist_cols] - off[hist_cols]) / (1 + np.abs(off[hist_cols])), nan=0.0).max(); max_hist = max(max_hist, float(d))
        if split[r] > 0:                                                         # encodings are out-of-fold in train, full-train in val/test
            max_enc = max(max_enc, float(np.nan_to_num(np.abs(on[enc_cols] - off[enc_cols]), nan=0.0).max()))
            bad_nan += int(np.any(np.isnan(on[enc_cols]) != np.isnan(off[enc_cols])))
        if int(r) in pos_in_te:
            p_on = parity.predict_proba(f"s{s}", ucid_str[raw_ex[r]], float(raw_ts[r]))
            max_p = max(max_p, abs(p_on - PRED[FINAL_NAME]["test"][pos_in_te[int(r)]])); n_te_rows += 1
        parity.observe(f"s{s}", ucid_str[raw_ex[r]], int(y_all[r]), float(raw_ts[r])); n_rows += 1
    for c in np.unique(raw["concept"].to_numpy()[rows]):                          # BKT mastery after the full history
        u = ucid_str[raw_ex[rows[raw["concept"].to_numpy()[rows] == c][0]]]
        max_bkt = max(max_bkt, abs(parity.bkt_mastery(f"s{s}", u) - float(mast_off[(uuid_values[s], int(c))])))
record("Online predictor reproduces offline features (train/val/test rows)", max_hist < 1e-4 and max_enc < 1e-5 and bad_nan == 0,
       f"{len(par_students)} students, {n_rows:,} rows, max rel. feature err {max_hist:.1e}, encoding err {max_enc:.1e}, NaN-pattern mismatches {bad_nan}")
record("Online predictor reproduces offline test probabilities", max_p < 1e-5, f"{n_te_rows:,} test rows, max |diff| {max_p:.1e}")
record("Online BKT mastery == offline BKT mastery", max_bkt < 1e-6, f"max |diff| {max_bkt:.1e}")
tick("reload + parity tests done")

PASS  Online predictor reproduces offline features (train/val/test rows)  [40 students, 3,818 rows, max rel. feature err 2.0e-08, encoding err 3.0e-08, NaN-pattern mismatches 0]
PASS  Online predictor reproduces offline test probabilities  [1,494 test rows, max |diff| 0.0e+00]
PASS  Online BKT mastery == offline BKT mastery  [max |diff| 5.0e-07]
[  684.9s] reload + parity tests done


## 11. Auto-generated report, environment record and hand-off bundle

In [21]:
def md_table(df, cols=None, digits=4, pct=()):
    cols = cols or list(df.columns); out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df[cols].iterrows():
        cells = []
        for c in cols:
            v = r[c]
            cells.append(f"{v:.2%}" if c in pct and isinstance(v, (float, np.floating)) else (f"{v:.{digits}f}" if isinstance(v, (float, np.floating)) else str(v)))
        out.append("| " + " | ".join(cells) + " |")
    return "\n".join(out)

tt = test_tbl.reset_index()
head_cols = ["model", "accuracy", "majority_accuracy_same_rows", "accuracy_gain_vs_majority_pp", "roc_auc", "pr_auc", "log_loss", "ece"]
fin = test_tbl.loc[FINAL_NAME]; base = test_tbl.loc["Majority / base-rate"]
ci = boot_tbl[boot_tbl["vs"] == "Majority / base-rate"].iloc[0]
real = (DATA_SOURCE == "junyi-real")
loose_name, strict_a = "HistGB-A LOOSE (sensitivity only)", next(iter(HGB_CONFIGS))
sens = ""
if loose_name in test_tbl.index:
    a_, b_ = test_tbl.loc[strict_a], test_tbl.loc[loose_name]
    sens = (f"\n**Sensitivity check.** On test, the loose variant scores ROC-AUC {b_['roc_auc']:.4f} / accuracy {b_['accuracy']:.2%} versus {a_['roc_auc']:.4f} / {a_['accuracy']:.2%} "
            f"for the strict version of the same configuration ({b_['roc_auc']-a_['roc_auc']:+.4f} AUC). The gap shows how much apparent signal depends on rows seeing same-timestamp history of unknown order; "
            "treat the loose number as an upper limit, never as a result.\n")
banner = "" if real else "\n> **WARNING: this report was produced from SYNTHETIC smoke-test data. None of the numbers below are results.**\n"
audit_lines = "\n".join(f"- {'PASS' if v['passed'] else '**FAIL**'} — {k}" + (f" ({v['detail']})" if v["detail"] else "") for k, v in AUDIT.items())

report = f"""# Member 2 — Final Report: Student Performance Prediction and Knowledge Tracing
{banner}
Generated {dt.datetime.now(dt.timezone.utc):%Y-%m-%d %H:%M UTC} by `member2_final_pipeline.ipynb` · data source: **{DATA_SOURCE}** · seed {SEED}

## 1. Headline result
Selected model: **{FINAL_NAME}** (lowest validation log-loss among strict learned models).
On the untouched **test period** it reaches **{fin['accuracy']:.2%} accuracy** against **{maj_te:.2%}** for the majority baseline on the same {len(y_te):,} rows
(**{fin['accuracy_gain_vs_majority_pp']:+.2f} percentage points**; student-bootstrap 95 % CI {ci['acc_ci95_low_pp']:+.2f} to {ci['acc_ci95_high_pp']:+.2f} pp),
ROC-AUC **{fin['roc_auc']:.4f}**, PR-AUC {fin['pr_auc']:.4f}, log-loss **{fin['log_loss']:.4f}** (base-rate predictor: {base['log_loss']:.4f}), ECE {fin['ece']:.4f}.
ROC-AUC is the primary ranking metric and log-loss the primary probability-quality metric, because accuracy depends heavily on the class balance of the period.

{md_table(tt, head_cols, 4)}

*(Threshold 0.5 for every model. The correct-answer rate drifts between periods — validation {y_val.mean():.2%}, test {y_te.mean():.2%} — so accuracy must always be compared with the majority baseline of the same rows.)*

## 2. Data and split
Junyi Academy `Log_Problem.csv` + `Info_Content.csv`; {len(raw):,} clean interactions, {raw['student'].nunique():,} students, {len(ucid_values):,} exercises in the log ({n_ex:,} in the exercise table).
Chronological split by timestamp value (70/15/15):

{md_table(split_summary, ['split','rows','students','start_utc','end_utc','correct_rate','rows_from_unseen_students'], 4)}

Timestamp resolution in the data is {bucket_seconds} s, so rows sharing a timestamp have no reliable order.

## 3. Features and leakage controls
Target: `is_correct` of the interaction being predicted. Features (all available *before* answering): {', '.join('`'+f+'`' for f in FEATURES)}.
History features use only the same student's answers from **strictly earlier timestamps**; one function builds them for train, validation and test (validation/test rows start from the student's accumulated history and keep updating it). `exercise_enc`/`concept_enc` are smoothed training-period correct rates (out-of-fold for training rows). Columns known only after answering ({', '.join(POST_ANSWER_COLUMNS)}) are never used.

Automated audit results:
{audit_lines}

## 4. Models compared (same strict features, same rows)
Majority/base-rate; student past-accuracy heuristic; Logistic Regression; HistGradientBoosting (two configurations, chosen on validation); BKT with guessed parameters and with parameters fitted on the training period ({', '.join(f'{k}={v:.3f}' for k, v in card['bkt_params'].items())}). A "loose" HistGB variant (rows may see earlier rows of their own timestamp bucket) is reported only as a sensitivity check and was never eligible for selection.
{sens}

### Validation (used for selection and thresholds)
{md_table(val_tbl.reset_index(), ['model','accuracy','majority_accuracy_same_rows','roc_auc','pr_auc','log_loss','brier','ece'], 4)}

### Test — final model at validation-tuned thresholds
{md_table(test_long[test_long.model == FINAL_NAME][['threshold_kind','threshold','accuracy','balanced_accuracy','precision','recall','precision_incorrect','recall_incorrect','roc_auc','log_loss']], None, 4)}

### Paired comparison (final model − reference, student-level bootstrap, 95 % CI)
{md_table(boot_tbl, ['vs','accuracy_diff_pp','acc_ci95_low_pp','acc_ci95_high_pp','logloss_diff','ll_ci95_low','ll_ci95_high'], 4)}

### Subgroups (test)
{md_table(subgroup_tbl, ['group','rows','correct_rate','majority_accuracy','accuracy','roc_auc','log_loss'], 4)}

## 5. Limitations (read before using the numbers)
1. **Conservative features.** Because timestamps are coarse, answers inside the same timestamp bucket are hidden. In production, same-session answers would be available; a validated session-aware feature (using `problem_number` order inside one exercise session) is a possible future improvement, but its ordering assumption could not be verified from this log.
2. **Temporal drift.** The correct-answer rate differs between validation and test; reported gains are relative to the majority baseline of the same rows.
3. **BKT** uses one global parameter set and `level4_id` as a concept proxy; it is a mastery tracker, not an optimised predictor. Mastery estimates in `bkt_mastery_estimates.csv.gz` are as of the end of the data.
4. **Cold start.** The first answers of a new student have no history; performance on them is lower (see subgroup table). Predictions for exercises never attempted rely on difficulty metadata and the training prior.
5. **Training sample.** Learned models were fitted on a {len(fit_idx):,}-row random sample of the {len(idx_tr):,} training rows. Features are history-based and were computed on all rows, so this does not introduce leakage.
6. **Single split.** One chronological hold-out; no repeated rolling-origin evaluation. Confidence intervals resample students, not time periods.
7. **No causal claim.** The model predicts correctness; it does not show that recommended exercises *cause* learning.
8. The final model is trained on the training period only. For deployment it can be refitted on all data with the same code.

## 6. Reproducibility
Seed {SEED}; libraries {json.dumps({k: v for k, v in VERSIONS.items() if v})}; BKT engine: {BKT_ENGINE}; HistGB/LR fit sample {len(fit_idx):,} rows.
Re-run: attach the Junyi dataset, `SMOKE_TEST=False`, **Save & Run All** in a fresh session. All artifacts are regenerated; `member2_manifest.json` lists SHA-256 hashes.

## 7. Using the model (Member 3)
```python
from learning_predictor import LearningPredictor
lp = LearningPredictor.from_files("member2_final_model.joblib", "member2_model_card.json", "member2_exercise_lookup.csv")
lp.observe(student_id, ucid, is_correct, ts)                 # after every answer, timestamps non-decreasing per student
p  = lp.predict_proba(student_id, ucid, ts)                  # P(correct) before answering
ps = lp.predict_many(student_id, candidate_ucids, ts)        # score many exercises at once
lp.rank_by_target(student_id, candidate_ucids, target=0.7)   # challenging-but-doable ordering
lp.bkt_mastery(student_id, ucid)                             # concept mastery in [0,1]
lp.predict_label(student_id, ucid, ts, mode="balanced")      # 1 = predicted correct (thresholds in the model card)
```
"""
(WORK / "member2_final_report.md").write_text(report, encoding="utf-8")
print(report[:2600], "\n...\n[full report written to member2_final_report.md]")

# Member 2 — Final Report: Student Performance Prediction and Knowledge Tracing

Generated 2026-10-23 19:29 UTC by `member2_final_pipeline.ipynb` · data source: **junyi-real** · seed 42

## 1. Headline result
Selected model: **HistGB-A (63 leaves, lr 0.10)** (lowest validation log-loss among strict learned models).
On the untouched **test period** it reaches **74.16% accuracy** against **70.71%** for the majority baseline on the same 2,432,333 rows
(**+3.44 percentage points**; student-bootstrap 95 % CI +3.19 to +3.72 pp),
ROC-AUC **0.7421**, PR-AUC 0.8658, log-loss **0.5244** (base-rate predictor: 0.6047), ECE 0.0045.
ROC-AUC is the primary ranking metric and log-loss the primary probability-quality metric, because accuracy depends heavily on the class balance of the period.

| model | accuracy | majority_accuracy_same_rows | accuracy_gain_vs_majority_pp | roc_auc | pr_auc | log_loss | ece |
|---|---|---|---|---|---|---|---|
| Majority / base-rate | 0.7071 | 0.7071 | 0.0000 | 0.5000 |

In [22]:
# ---------------- environment record, README, manifest, zip ----------------
env = {"data_source": DATA_SOURCE, "seed": SEED, "versions": VERSIONS, "bkt_engine": BKT_ENGINE, "platform": platform.platform(),
       "config": {"STUDENT_FRACTION": STUDENT_FRACTION, "HGB_TRAIN_ROWS": HGB_TRAIN_ROWS, "TUNE_HGB": TUNE_HGB, "RUN_LOOSE_SENSITIVITY": RUN_LOOSE_SENSITIVITY,
                  "BOOTSTRAP_B": BOOTSTRAP_B, "SMOKE_TEST": SMOKE_TEST},
       "rows": {"raw_processed": int(total_rows), "clean": int(len(raw))}, "runtime_seconds": round(time.time() - _t0, 1)}
json.dump(env, open(WORK / "environment.json", "w"), indent=2)
(WORK / "requirements_member2.txt").write_text("\n".join(f"{k}=={v}" for k, v in VERSIONS.items() if v and k != "python") + "\n")
(WORK / "README_handoff.md").write_text(f"""# Member 2 -> Member 3 hand-off
Final model: **{FINAL_NAME}**  (data source: {DATA_SOURCE})

Files: member2_final_model.joblib, member2_model_card.json, member2_exercise_lookup.csv, learning_predictor.py, bkt_mastery_estimates.csv.gz,
member2_final_metrics.csv, member2_validation_metrics.csv, member2_final_report.md.
Requirements: see requirements_member2.txt (scikit-learn {VERSIONS['scikit-learn']}; use the same major/minor version to unpickle).

Interface summary (full example in member2_final_report.md section 7):
  lp = LearningPredictor.from_files('member2_final_model.joblib','member2_model_card.json','member2_exercise_lookup.csv')
  lp.observe(student, ucid, is_correct, ts); lp.predict_proba(student, ucid, ts); lp.predict_many(student, ucids, ts); lp.bkt_mastery(student, ucid)

Behaviour you can rely on: unknown exercise ids / missing metadata never raise; brand-new students are handled; probabilities are in (0,1);
answers inside the current timestamp bucket are not visible to a prediction (same rule as training).
Thresholds for hard 0/1 labels are in the model card (balanced: {t_bal}, accuracy: {t_acc}).
""", encoding="utf-8")

bundle = ["member2_final_model.joblib", "member2_model_card.json", "member2_exercise_lookup.csv", "learning_predictor.py", "bkt_mastery_estimates.csv.gz",
          "member2_final_metrics.csv", "member2_validation_metrics.csv", "member2_bootstrap_ci.csv", "member2_subgroup_metrics.csv", "member2_threshold_table.csv",
          "member2_split_summary.csv", "member2_feature_importance.csv", "member2_audit_results.json", "member2_final_report.md", "README_handoff.md",
          "environment.json", "requirements_member2.txt", "member2_roc_pr.png", "member2_calibration.png", "member2_feature_importance.png"]
manifest = {f: {"bytes": (WORK / f).stat().st_size, "sha256": hashlib.sha256((WORK / f).read_bytes()).hexdigest()} for f in bundle}
json.dump(manifest, open(WORK / "member2_manifest.json", "w"), indent=2)
ZIP_PATH = WORK / "member2_final_handoff.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in bundle + ["member2_manifest.json"]: z.write(WORK / f, arcname=f)
print("Hand-off bundle:", ZIP_PATH.name, f"({ZIP_PATH.stat().st_size/1e6:.1f} MB)  with {len(bundle)+1} files")
display(FileLink(str(ZIP_PATH.name)))

Hand-off bundle: member2_final_handoff.zip (9.7 MB)  with 21 files


/kaggle/working/member2_final_handoff.zip

## 12. Final checklist (generated from this run)

In [23]:
status = [
    ("Student-history features identical for train / val / test (one function)", AUDIT.get("Same feature semantics in train / val / test (NaN only when there is no earlier answer)", {}).get("passed", False)),
    ("Strict features audited: independent recomputation + label-flip tests", AUDIT["Vectorised features == independent naive recomputation"]["passed"] and AUDIT["Label-flip test: flipping an answer never changes its own / same-timestamp / earlier / other-student features"]["passed"]),
    ("Chronological split verified; test scored once", AUDIT["Chronological split: train < validation < test (no shared timestamp)"]["passed"]),
    ("Majority + heuristic + LR + HistGB + BKT compared on the SAME rows", all(len(PRED[n]["test"]) == len(idx_te) for n in PRED)),
    ("Threshold tuned on validation only; accuracy, balanced, ROC-AUC, PR-AUC, log-loss, calibration reported", True),
    ("Final model saved, reloaded, prediction interface tested", AUDIT["Saved model reloads and reproduces in-memory predictions"]["passed"] and AUDIT["Prediction interface handles cold-start, unknown exercise, missing metadata, ts=None, batches, bad order"]["passed"]),
    ("Online predictor == offline pipeline (features, probabilities, BKT)", all(AUDIT[k]["passed"] for k in ["Online predictor reproduces offline features (train/val/test rows)", "Online predictor reproduces offline test probabilities", "Online BKT mastery == offline BKT mastery"])),
    ("Report, environment record, manifest and hand-off zip written", ZIP_PATH.exists()),
]
print(f"DATA SOURCE: {DATA_SOURCE}" + ("" if DATA_SOURCE == "junyi-real" else "   <-- SMOKE TEST: numbers are NOT results; re-run with SMOKE_TEST=False on the real data"))
print(f"FINAL MODEL: {FINAL_NAME}\n")
for name, passed in status: print(("[x] " if passed else "[ ] ") + name)
print("\nAll automated checks passed." if all(p for _, p in status) and all(v["passed"] for v in AUDIT.values())
      else "\nSOME CHECKS FAILED - read the FAIL lines above before submitting.")
print(f"Total runtime: {(time.time()-_t0)/60:.1f} min")

DATA SOURCE: junyi-real
FINAL MODEL: HistGB-A (63 leaves, lr 0.10)

[x] Student-history features identical for train / val / test (one function)
[x] Strict features audited: independent recomputation + label-flip tests
[x] Chronological split verified; test scored once
[x] Majority + heuristic + LR + HistGB + BKT compared on the SAME rows
[x] Threshold tuned on validation only; accuracy, balanced, ROC-AUC, PR-AUC, log-loss, calibration reported
[x] Final model saved, reloaded, prediction interface tested
[x] Online predictor == offline pipeline (features, probabilities, BKT)
[x] Report, environment record, manifest and hand-off zip written

All automated checks passed.
Total runtime: 11.4 min
